Step 0: Setup

In [ ]:
# Item 8: install pinned versions when a requirements.txt is present, otherwise install
# unpinned and write the resolved versions out at the end of Step 0.
import os, subprocess, sys
from pathlib import Path

_REQ = Path("requirements.txt")
if _REQ.exists():
    print("Installing from pinned requirements.txt")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_REQ)], check=False)
else:
    print("No requirements.txt found - installing unpinned, versions are frozen at the end of Step 0.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "earthengine-api", "numpy", "pandas", "matplotlib",
                    "scikit-image", "scipy", "torch", "torchvision"], check=False)

In [ ]:
from pathlib import Path
import os, sys, math, json, random, platform, io, time, urllib.request
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---- Item 9: guard Colab-only imports so this notebook also runs in local Jupyter ----
IN_COLAB = "google.colab" in sys.modules
colab_files = colab_drive = None
if IN_COLAB:
    from google.colab import files as colab_files      # noqa: F401
    from google.colab import drive as colab_drive      # noqa: F401

def maybe_mount_drive():
    """Item 2: mounting Drive is opt-in and never required. Off Colab this is a no-op.
    Set CVSR_MOUNT_DRIVE=1 to mount; nothing downstream depends on a Drive path."""
    if not IN_COLAB or os.environ.get("CVSR_MOUNT_DRIVE", "0") != "1":
        return None
    colab_drive.mount("/content/drive")
    return Path("/content/drive/MyDrive")

DRIVE_ROOT = maybe_mount_drive()

# Optional extras (used if present, skipped cleanly if not)
try:
    from skimage.metrics import structural_similarity as ssim
    SKIMAGE_AVAILABLE = True
except Exception:
    SKIMAGE_AVAILABLE = False
try:
    from scipy import stats as scipy_stats
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

# display() works in notebooks; define a no-op-safe fallback for plain Python
try:
    display
except NameError:
    from IPython.display import display

print("Python      :", sys.version.split()[0], "|", platform.machine())
print("PyTorch     :", torch.__version__)
print("Environment :", "Colab" if IN_COLAB else "local Jupyter/Python")
print("Drive       :", DRIVE_ROOT if DRIVE_ROOT else "not mounted (not required)")
print("CUDA avail  :", torch.cuda.is_available())
print("MPS avail   :", torch.backends.mps.is_available() if hasattr(torch.backends, "mps") else False)
print("scikit-image:", SKIMAGE_AVAILABLE, "| scipy:", SCIPY_AVAILABLE)

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device: CUDA (Colab) -> MPS (Apple Silicon) -> CPU
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
def choose_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")
device = choose_device()
try:
    _ = torch.ones(2, device=device)
except Exception as e:
    print("Device smoke test failed, falling back to CPU:", e)
    device = torch.device("cpu")
print("Using device:", device)

In [ ]:
import ee

# Item 2: read the Earth Engine project from the environment so no private project ID
# is committed to the public repository. Set EE_PROJECT before running, e.g.
#   os.environ["EE_PROJECT"] = "my-ee-project"
EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()

EE_READY = False
if not EE_PROJECT:
    print("EE_PROJECT environment variable not set - Earth Engine NOT initialized.")
    print("That is fine if the baseline CSV and SR pairs are already cached on disk.")
else:
    try:
        ee.Initialize(project=EE_PROJECT)
        EE_READY = True
    except Exception:
        try:
            ee.Authenticate()
            ee.Initialize(project=EE_PROJECT)
            EE_READY = True
        except Exception as e:
            print("Earth Engine NOT initialized:", str(e)[:160])
            print("That is fine if the baseline CSV and SR pairs are already cached on disk.")
    if EE_READY:
        print("Earth Engine initialized.")

Step 2: Global Configuration

Defines variables for rest of notebook

In [ ]:
# ---------- Paths (Item 2: no hard-coded Drive paths; all overridable by env var) ----------
PROJECT_ROOT  = Path(os.environ.get("CVSR_PROJECT_ROOT", ".")).resolve()
PAIRS_DIR     = Path(os.environ.get("CVSR_PAIRS_DIR", PROJECT_ROOT / "data_pairs" / "sr_pairs_x4"))
RESULTS_DIR   = Path(os.environ.get("CVSR_RESULTS_DIR", PROJECT_ROOT / "results"))
FIG_DIR       = Path(os.environ.get("CVSR_FIG_DIR", PROJECT_ROOT / "figures" / "paper"))
CHECKPOINT_DIR= Path(os.environ.get("CVSR_CKPT_DIR", PROJECT_ROOT / "checkpoints"))
PAPER_DIR     = Path(os.environ.get("CVSR_PAPER_DIR", PROJECT_ROOT / "paper"))
FROZEN_DIR    = Path(os.environ.get("CVSR_FROZEN_DIR", PROJECT_ROOT / "frozen_artifacts"))
for d in [PAIRS_DIR, RESULTS_DIR, FIG_DIR, CHECKPOINT_DIR, PAPER_DIR, FROZEN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---------- Item 3: explicit cache workflow ----------
# USE_CACHED_PAIRS=False  -> generate pairs into PAIRS_DIR (default; what a fresh user gets)
# USE_CACHED_PAIRS=True   -> read pairs from CACHED_PAIRS_DIR and skip generation entirely
USE_CACHED_PAIRS = os.environ.get("CVSR_USE_CACHED_PAIRS", "0") == "1"
_cached_env      = os.environ.get("CVSR_CACHED_PAIRS_DIR", "").strip()
CACHED_PAIRS_DIR = Path(_cached_env) if _cached_env else None
RESET_PAIRS      = False   # Item 5: True wipes PAIRS_DIR before generating (requires confirmation)
AUTO_DOWNLOAD    = False   # Item 4: True re-enables Colab auto-download of pair archives

if USE_CACHED_PAIRS and CACHED_PAIRS_DIR is None:
    raise ValueError("USE_CACHED_PAIRS is on but CVSR_CACHED_PAIRS_DIR is not set. "
                     "Point it at a directory of .npz pairs, or set USE_CACHED_PAIRS=False.")
# Every downstream read of the pair dataset goes through PAIRS_SOURCE.
PAIRS_SOURCE = CACHED_PAIRS_DIR if USE_CACHED_PAIRS else PAIRS_DIR

TRAIN_CSV = RESULTS_DIR / "train_files.csv"
VAL_CSV   = RESULTS_DIR / "val_files.csv"
TEST_CSV  = RESULTS_DIR / "test_files.csv"
BASELINE_CSV = RESULTS_DIR / "baseline_ndvi_all_crops.csv"

# ---------- Study area / period ----------
S2_SR_ID  = "COPERNICUS/S2_SR_HARMONIZED"
CDL_ID    = "USDA/NASS/CDL"
AOI_BBOX  = [-122.0, 38.0, -121.0, 38.5]          # Northern Central Valley / Sacramento-Delta
YEARS     = [2018, 2019, 2020, 2021, 2022]
# Item 11: year groups are DERIVED from real PRISM annual precipitation relative to the
# 2018-2022 study-period mean, and are named higher-/lower-precipitation. They are a
# relative grouping, NOT a climatological drought classification. See Step 1.
CROP_CODES  = {"ALFALFA": 36, "ALMOND": 75, "CITRUS": 72, "GRAPES": 69}  # USDA CDL codes
CROP_CODE_LIST = list(CROP_CODES.values())
TARGET_CROPS   = list(CROP_CODES.keys())

# Crops the MANUSCRIPT explicitly excludes from all super-resolution claims.
# Citrus is included in the CDL target-class union at patch-generation time, but citrus is
# uncommon in this AOI and no patch in the final dataset resolves citrus as its dominant
# crop, so there are zero citrus SR patches and zero citrus test patches. SR performance
# therefore cannot be claimed for citrus at all. Citrus REMAINS valid in the baseline NDVI
# analysis, which is computed from CDL-masked scene means and does not depend on patches.
# The readiness check fails if a crop is absent from the SR data but missing from this list.
SR_CROPS_EXCLUDED_FROM_CLAIMS = ["CITRUS"]

PRECIP_GROUP_HIGH = "higher-precipitation"
PRECIP_GROUP_LOW  = "lower-precipitation"
# maps any legacy label found in a cached CSV onto the current naming
LEGACY_LABEL_MAP = {"wet": PRECIP_GROUP_HIGH, "drought": PRECIP_GROUP_LOW,
                    PRECIP_GROUP_HIGH: PRECIP_GROUP_HIGH, PRECIP_GROUP_LOW: PRECIP_GROUP_LOW}

# ---------- Bands & NDVI indices (FIXED ONCE - do not redefine) ----------
SR_BANDS = ["B4", "B3", "B2", "B8"]   # Red, Green, Blue, NIR
RED_IDX  = 0                          # B4
NIR_IDX  = 3                          # B8

# ---------- Super-resolution settings ----------
SCALE           = 4
PATCH_SIZE_HR   = 128                 # divisible by SCALE
PATCH_SIZE_LR   = PATCH_SIZE_HR // SCALE
HR_SCALE_METERS = 10                  # S2 native res for these bands
CDL_SCALE       = 30                  # CDL native res
# Item: this threshold applies to the UNION of the target crop classes, not to a single
# crop. A patch qualifies if >= 30% of its CDL pixels belong collectively to any target
# crop; the dominant crop is recorded separately in manifest_details.csv.
MIN_CROP_FRACTION = 0.3
COMPOSITE_START_MONTHDAY = "06-01"    # growing-season composite window used for SR patches
COMPOSITE_END_MONTHDAY   = "09-30"

# ---------- Cloud masking / baseline NDVI (Items 13, 14) ----------
MAX_CLOUD = 60                        # CLOUDY_PIXEL_PERCENTAGE ceiling; state this in Methods
# Item 13: SCL 4/5/6 are vegetation, bare soil and water. Class 7 is UNCLASSIFIED /
# LOW-PROBABILITY CLOUD in Sentinel-2 L2A - it is NOT a guaranteed clear-sky class. It is
# retained here to preserve scene coverage, and the residual contamination risk is
# disclosed as a limitation. Set KEEP_SCL = (4, 5, 6) for stricter masking, which requires
# regenerating the pairs and retraining.
KEEP_SCL = (4, 5, 6, 7)
NDVI_REDUCE_SCALE = 60                # metres; the baseline regional mean is reduced at 60 m,
                                      # NOT at the 10 m native band resolution. Say 60 m in Methods.

# ---------- Train / validation / test split ----------
# Primary strategy: chronological hold-out by acquisition year. This prevents same-composite
# leakage -- nearby/overlapping 1.28 km patches drawn from the same growing-season composite
# could otherwise appear on both sides of a random split. Because the same geographic fields
# recur across years, it tests temporal generalisation within one region rather than full
# spatial independence.
SPLIT_TRAIN_YEARS  = [2018, 2019, 2020]
SPLIT_VAL_YEAR     = 2021
SPLIT_TEST_YEAR    = 2022
GEO_BLOCK_SIZE_DEG      = 0.02        # ~2.2 km blocks at this latitude, bigger than the 1.28 km patch
SPATIAL_SPLIT_FRACTIONS = {"train": 0.6, "val": 0.2, "test": 0.2}
MIN_TEST_CROP_COUNT     = 30
MIN_TEST_CROP_SHARE     = 0.05

# ---------- Display-only figure settings (Items 16, 17, 18, 19) ----------
DISPLAY_STRETCH  = (0.0, 0.30)        # fixed reflectance stretch for RGB panels, DISPLAY ONLY
NDVI_VMIN, NDVI_VMAX = 0.0, 1.0       # shared NDVI colour scale
NDVI_ERR_VMAX    = 0.15               # fixed NDVI-error scale across every example
N_FIGURE_EXAMPLES = 3
EXAMPLE_SELECTION_RULE = "median_psnr"   # "median_psnr" | "quartiles" | "seeded_random"

# ---------- Run controls ----------
QUICK_TEST           = False
NUM_PATCHES_PER_YEAR = 3000
MIN_TOTAL_PAIRS      = 15000
MAX_ATTEMPTS_FACTOR  = 10

REGENERATE_BASELINE  = False
REGENERATE_PAIRS     = False
REGENERATE_MANIFEST_DETAILS = False
REGENERATE_PRECIP_LABELS    = False

assert PATCH_SIZE_HR % SCALE == 0, "PATCH_SIZE_HR must be divisible by SCALE"
AOI = ee.Geometry.Rectangle(AOI_BBOX) if EE_READY else None

print("Bands      :", SR_BANDS, "| RED_IDX", RED_IDX, "(B4) NIR_IDX", NIR_IDX, "(B8)")
print("Crops      :", CROP_CODES)
print("SR         : x{} | HR {} | LR {}".format(SCALE, PATCH_SIZE_HR, PATCH_SIZE_LR))
print("Cloud/SCL  : CLOUDY_PIXEL_PERCENTAGE <=", MAX_CLOUD, "| KEEP_SCL", KEEP_SCL)
print("NDVI scale :", NDVI_REDUCE_SCALE, "m (regional reduction, not band resolution)")
print("Pairs read :", PAIRS_SOURCE, "(cached)" if USE_CACHED_PAIRS else "(generated here)")

In [ ]:
# Provenance manifest: records the exact configuration that produced every frozen output.
run_manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "project_root": str(PROJECT_ROOT),
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "seed": SEED,
    "device": str(device),
    "aoi_bbox": AOI_BBOX,
    "years": YEARS,
    "band_order": SR_BANDS,
    "red_idx": RED_IDX, "nir_idx": NIR_IDX,
    # science configuration, so Methods can be written straight from this file
    "cloudy_pixel_percentage_max": MAX_CLOUD,
    "keep_scl_classes": list(KEEP_SCL),
    "scl_7_note": "SCL 7 is unclassified/low-probability cloud, retained for coverage; "
                  "residual contamination disclosed as a limitation",
    "ndvi_reduce_scale_m": NDVI_REDUCE_SCALE,
    "baseline_ndvi_unit": "per-scene regional mean over crop-masked pixels (not monthly medians)",
    "sr_composite_window": f"{COMPOSITE_START_MONTHDAY} to {COMPOSITE_END_MONTHDAY} median composite",
    "crop_coverage_threshold": MIN_CROP_FRACTION,
    "crop_coverage_threshold_note": "applies to the union of target crop classes, not a single crop",
    "patch_size_hr": PATCH_SIZE_HR, "patch_size_lr": PATCH_SIZE_LR, "scale_factor": SCALE,
    "use_cached_pairs": USE_CACHED_PAIRS,
    "pairs_source_dir": str(PAIRS_SOURCE),
    "baseline_data_source": None,
    "sr_pair_data_source": None,
    "paper_ready": False,
    "notes": [],
}
def save_manifest():
    with open(RESULTS_DIR / "run_manifest.json", "w") as f:
        json.dump(run_manifest, f, indent=2, default=str)
save_manifest()
print("Manifest initialised:", RESULTS_DIR / "run_manifest.json")

Step 2b: freeze package versions

Writes requirements.txt with the exact versions this run used. Re-running Step 0 with that file present installs the pinned set.

In [ ]:
# Item 8: freeze the exact package versions this run used.
import importlib.metadata as importlib_metadata

PKGS = ["earthengine-api", "numpy", "pandas", "matplotlib", "scikit-image",
        "scipy", "torch", "torchvision"]
lines_req = [f"# generated {datetime.now().isoformat(timespec='seconds')}",
             f"# python=={sys.version.split()[0]}"]
resolved = {}
for p in PKGS:
    try:
        v = importlib_metadata.version(p)
        resolved[p] = v
        lines_req.append(f"{p}=={v}")
    except importlib_metadata.PackageNotFoundError:
        lines_req.append(f"# {p} not installed in this environment")

req_path = PROJECT_ROOT / "requirements.txt"
req_path.write_text("\n".join(lines_req) + "\n")
run_manifest["package_versions"] = resolved
save_manifest()
print("Wrote", req_path)
print("\n".join(lines_req))
print("\nCommit this file. Re-running Step 0 with it present installs the pinned versions.")

PRISM Annual precipitation labels

Output: results/annual_precipitation_labels.csv with columns year, annual_precipitation_mm, multi_year_mean_mm, label.

In [ ]:
PRECIP_LABELS_CSV = RESULTS_DIR / "annual_precipitation_labels.csv"

def prism_annual_precip_mm(year, region, scale_m=4000):
    """Real annual precipitation (mm) for `year`, averaged over `region`, from PRISM ANm."""
    start = f"{year}-01-01"; end = f"{year}-12-31"
    annual_total = (ee.ImageCollection("OREGONSTATE/PRISM/ANm")
                    .filterDate(start, end).select("ppt").sum())
    val = annual_total.reduceRegion(reducer=ee.Reducer.mean(), geometry=region,
                                    scale=scale_m, bestEffort=True, maxPixels=int(1e9)).get("ppt")
    return float(ee.Number(val).getInfo())

if PRECIP_LABELS_CSV.exists() and not REGENERATE_PRECIP_LABELS:
    precip_df = pd.read_csv(PRECIP_LABELS_CSV)
    run_manifest["year_labels_source"] = f"cached_csv:{PRECIP_LABELS_CSV}"
    print("Loaded cached PRISM precipitation labels:", PRECIP_LABELS_CSV)
else:
    if not EE_READY:
        raise RuntimeError("annual_precipitation_labels.csv must be built from Earth Engine (PRISM), "
                           "but EE is not initialized. Set EE_PROJECT, or provide a cached CSV.")
    print("Computing real annual precipitation from PRISM for", YEARS, "...")
    rows = [{"year": y, "annual_precipitation_mm": prism_annual_precip_mm(y, AOI)} for y in YEARS]
    precip_df = pd.DataFrame(rows).sort_values("year").reset_index(drop=True)
    study_mean = float(precip_df["annual_precipitation_mm"].mean())
    precip_df["study_period_mean_mm"] = study_mean
    precip_df["anomaly_mm"] = precip_df["annual_precipitation_mm"] - study_mean
    # Item 11: a RELATIVE grouping against the 2018-2022 study-period mean. This is not a
    # climatological drought index and must not be described as one.
    precip_df["precip_group"] = np.where(
        precip_df["annual_precipitation_mm"] >= study_mean, PRECIP_GROUP_HIGH, PRECIP_GROUP_LOW)
    precip_df.to_csv(PRECIP_LABELS_CSV, index=False)
    run_manifest["year_labels_source"] = "earth_engine_real:PRISM/ANm"
    print("Saved PRISM annual precipitation groups:", PRECIP_LABELS_CSV)

# accept a legacy cached CSV that still uses wet/drought column names
if "precip_group" not in precip_df.columns and "label" in precip_df.columns:
    precip_df["precip_group"] = precip_df["label"].str.lower().map(LEGACY_LABEL_MAP)
if "study_period_mean_mm" not in precip_df.columns and "multi_year_mean_mm" in precip_df.columns:
    precip_df["study_period_mean_mm"] = precip_df["multi_year_mean_mm"]
if "anomaly_mm" not in precip_df.columns:
    precip_df["anomaly_mm"] = precip_df["annual_precipitation_mm"] - precip_df["study_period_mean_mm"]
precip_df = precip_df[["year", "annual_precipitation_mm", "study_period_mean_mm",
                       "anomaly_mm", "precip_group"]]
precip_df.to_csv(PRECIP_LABELS_CSV, index=False)

YEAR_GROUPS = dict(zip(precip_df["year"].astype(int), precip_df["precip_group"]))
run_manifest["year_precip_groups"] = YEAR_GROUPS
run_manifest["precip_grouping_method"] = (
    "annual PRISM precipitation vs the 2018-2022 study-period mean; relative grouping, "
    "not a climatological drought index")
save_manifest()
display(precip_df.round(2))
print("Year groups (derived from PRISM):", YEAR_GROUPS)
print("NOTE: these are relative precipitation groups, not a drought classification.")

Step 2: EE Helpers

In [ ]:
def mask_s2_scl(img):
    """Cloud-mask Sentinel-2 L2A using the SCL band, keeping the classes in KEEP_SCL.

    KEEP_SCL = (4, 5, 6, 7): vegetation, bare soil, water, and unclassified.
    Class 7 is UNCLASSIFIED / LOW-PROBABILITY CLOUD -- it is not a confirmed clear-sky
    class. It is retained to preserve scene coverage, and the resulting residual
    contamination risk is disclosed in the manuscript limitations."""
    scl = img.select("SCL")
    good = scl.eq(int(KEEP_SCL[0]))
    for c in KEEP_SCL[1:]:
        good = good.Or(scl.eq(int(c)))
    return img.updateMask(good)

def get_cdl_image(year):
    """USDA CDL image for a given year."""
    img = ee.ImageCollection(CDL_ID).filter(ee.Filter.calendarRange(year, year, "year")).first()
    return ee.Image(img)

def cdl_crop_band_name(cdl_img):
    """CDL band name varies by year; pick 'cropland'/'classification' or the first band."""
    bands = cdl_img.bandNames()
    return ee.String(ee.Algorithms.If(bands.contains("cropland"), "cropland",
                ee.Algorithms.If(bands.contains("classification"), "classification", bands.get(0))))

def cdl_is_crop(year, codes):
    """Binary layer: 1 where CDL is in `codes`, else 0."""
    cdl  = get_cdl_image(year)
    band = cdl_crop_band_name(cdl)
    return cdl.select([band]).remap(list(codes), [1]*len(codes), 0).rename("is_crop")

def seasonal_composite(year):
    """Cloud-masked GROWING-SEASON (June 1 - September 30) median composite of the SR bands.

    SR patches are extracted from these seasonal composites -- they are NOT sampled
    year-round. Methods must describe the composite window, not a year-round sample."""
    start = f"{year}-{COMPOSITE_START_MONTHDAY}"; end = f"{year}-{COMPOSITE_END_MONTHDAY}"
    col = (ee.ImageCollection(S2_SR_ID)
           .filterBounds(AOI).filterDate(start, end)
           .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD))
           .map(mask_s2_scl))
    return col.select(SR_BANDS).median()

print("Earth Engine helpers ready." if EE_READY else "Earth Engine helpers defined (EE not initialized).")
print(f"Cloud filter: CLOUDY_PIXEL_PERCENTAGE <= {MAX_CLOUD} | SCL kept: {KEEP_SCL}")

Step 3: Baseline Crop Specific NDVI (Sentinel-2 & CDL)

Note: citrus is uncommon in northern AOI; crops with insufficient data are reported as a not analysable

In [ ]:
def ndvi_timeseries_for_crop(year, crop_code, scale=NDVI_REDUCE_SCALE):
    """Real per-scene mean NDVI over one crop's CDL pixels for one year."""
    crop_mask = cdl_is_crop(year, [crop_code]).selfMask()   # 1 on this crop, masked elsewhere
    start = f"{year}-01-01"; end = f"{year}-12-31"
    col = (ee.ImageCollection(S2_SR_ID)
           .filterBounds(AOI).filterDate(start, end)
           .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD))
           .map(mask_s2_scl))
    def add_ndvi(im):
        nir = im.select("B8"); red = im.select("B4")
        ndvi = nir.subtract(red).divide(nir.add(red).add(1e-6)).rename("NDVI")
        return im.addBands(ndvi).updateMask(crop_mask)
    col = col.map(add_ndvi)
    def reduce_one(im):
        v = im.select("NDVI").reduceRegion(
            reducer=ee.Reducer.mean(), geometry=AOI,
            scale=scale, bestEffort=True, maxPixels=int(1e9), tileScale=4
        ).get("NDVI")
        return ee.Feature(None, {"date": im.date().format("YYYY-MM-dd"), "ndvi_mean": v})
    fc = ee.FeatureCollection(col.map(reduce_one)).filter(ee.Filter.notNull(["ndvi_mean"]))
    feats = fc.getInfo()["features"]
    return [(f["properties"]["date"], f["properties"]["ndvi_mean"]) for f in feats]

In [ ]:
if BASELINE_CSV.exists() and not REGENERATE_BASELINE:
    baseline_df = pd.read_csv(BASELINE_CSV, parse_dates=["date"])
    run_manifest["baseline_data_source"] = f"cached_csv:{BASELINE_CSV}"
    print("Loaded cached baseline NDVI:", BASELINE_CSV, "| rows:", len(baseline_df))
else:
    if not EE_READY:
        raise RuntimeError("Baseline NDVI must be built from Earth Engine, but EE is not initialized. "
                           "Set EE_PROJECT in Step 0, or provide a cached baseline_ndvi_all_crops.csv.")
    print("Building baseline NDVI from Earth Engine (this can take several minutes)...")
    rows = []
    for crop_name, code in CROP_CODES.items():
        crop_total = 0
        for year in YEARS:
            try:
                series = ndvi_timeseries_for_crop(year, code)
            except Exception as e:
                print(f"   {crop_name} {year}: EE error ({str(e)[:80]}) - skipped")
                continue
            for date_str, val in series:
                rows.append({"date": date_str, "year": year, "crop": crop_name,
                             "ndvi_mean": float(val),
                             "precip_group": YEAR_GROUPS.get(year, "unknown")})
            crop_total += len(series)
        print(f"   {crop_name}: {crop_total} scene-observations across {len(YEARS)} years")
    baseline_df = pd.DataFrame(rows)
    if baseline_df.empty:
        raise ValueError("No baseline NDVI observations returned. Check AOI/crop codes/EE access.")
    baseline_df["date"] = pd.to_datetime(baseline_df["date"])
    baseline_df.to_csv(BASELINE_CSV, index=False)
    run_manifest["baseline_data_source"] = f"earth_engine_real:{S2_SR_ID}+{CDL_ID}"
    print("Saved baseline NDVI:", BASELINE_CSV, "| rows:", len(baseline_df))

# Item 11: normalise any legacy wet/drought labels in a cached CSV onto the current naming
if "precip_group" not in baseline_df.columns and "label" in baseline_df.columns:
    baseline_df["precip_group"] = baseline_df["label"].str.lower().map(LEGACY_LABEL_MAP)
    print("Legacy wet/drought labels remapped to higher-/lower-precipitation.")
baseline_df["precip_group"] = baseline_df["year"].map(YEAR_GROUPS).fillna(baseline_df["precip_group"])
baseline_df = baseline_df.drop(columns=[c for c in ["label"] if c in baseline_df.columns])
baseline_df.to_csv(BASELINE_CSV, index=False)

run_manifest["baseline_n_scene_observations"] = int(len(baseline_df))
save_manifest()
display(baseline_df.head())
print("Crops present:", sorted(baseline_df["crop"].unique()))
print("Groups present:", sorted(baseline_df["precip_group"].dropna().unique()))
print("NOTE: each row is ONE Sentinel-2 scene's regional mean NDVI over that crop's CDL "
      f"pixels, reduced at {NDVI_REDUCE_SCALE} m. These are not monthly composites.")

Baseline NDVI summary

Saved at content/results/baseline_NDVI_summary.csv

In [ ]:
# Item 12: the Welch t-tests have been REMOVED.
# Hundreds of scene-level regional means from a single AOI across repeated dates are
# temporally autocorrelated -- they are not independent biological replicates, so a
# scene-level t-test overstates certainty. This cell reports group means, standard
# deviations, differences, percent changes and observation counts only.
# A year-aggregated companion table is written alongside it for anyone who wants a
# defensible temporal unit (n = 5 annual means per crop, still descriptive).

summary_rows = []
for crop, sub in baseline_df.groupby("crop"):
    high = sub[sub["precip_group"].eq(PRECIP_GROUP_HIGH)]["ndvi_mean"].dropna()
    low  = sub[sub["precip_group"].eq(PRECIP_GROUP_LOW)]["ndvi_mean"].dropna()
    high_mean = high.mean() if len(high) else np.nan
    low_mean  = low.mean()  if len(low)  else np.nan
    diff = low_mean - high_mean
    pct  = 100 * diff / high_mean if (high_mean and not np.isnan(high_mean)) else np.nan
    summary_rows.append({
        "crop": crop,
        "higher_precip_mean_ndvi": high_mean,
        "higher_precip_sd_ndvi": high.std(ddof=1) if len(high) > 1 else np.nan,
        "lower_precip_mean_ndvi": low_mean,
        "lower_precip_sd_ndvi": low.std(ddof=1) if len(low) > 1 else np.nan,
        "lower_minus_higher": diff,
        "percent_change": pct,
        "n_higher_precip": int(len(high)),
        "n_lower_precip": int(len(low)),
    })
baseline_summary = pd.DataFrame(summary_rows).sort_values("crop").reset_index(drop=True)
baseline_summary.to_csv(RESULTS_DIR / "baseline_ndvi_summary.csv", index=False)
display(baseline_summary.round(4))

# year-aggregated companion: one mean per crop per year, the defensible temporal unit
annual_means = (baseline_df.groupby(["crop", "year", "precip_group"])["ndvi_mean"]
                .agg(annual_mean_ndvi="mean", n_scenes="size").reset_index())
annual_means.to_csv(RESULTS_DIR / "baseline_ndvi_annual_means.csv", index=False)

annual_rows = []
for crop, sub in annual_means.groupby("crop"):
    high = sub[sub["precip_group"].eq(PRECIP_GROUP_HIGH)]["annual_mean_ndvi"]
    low  = sub[sub["precip_group"].eq(PRECIP_GROUP_LOW)]["annual_mean_ndvi"]
    annual_rows.append({
        "crop": crop,
        "higher_precip_annual_mean": high.mean() if len(high) else np.nan,
        "lower_precip_annual_mean": low.mean() if len(low) else np.nan,
        "difference": (low.mean() - high.mean()) if (len(high) and len(low)) else np.nan,
        "n_years_higher": int(len(high)), "n_years_lower": int(len(low)),
    })
baseline_annual_summary = pd.DataFrame(annual_rows).sort_values("crop").reset_index(drop=True)
baseline_annual_summary.to_csv(RESULTS_DIR / "baseline_ndvi_annual_summary.csv", index=False)
display(baseline_annual_summary.round(4))

run_manifest["baseline_inferential_tests"] = (
    "none - removed; scene-level observations are temporally autocorrelated. "
    "Descriptive means, SDs, differences and counts only.")
save_manifest()

weak = baseline_summary[(baseline_summary["n_higher_precip"] < 3) |
                        (baseline_summary["n_lower_precip"] < 3)]["crop"].tolist()
if weak:
    print("NOTE: insufficient data in this AOI to interpret:", weak,
          "- report these as not analysable rather than drawing conclusions.")
print("\nReport these as descriptive group differences. Do not attach p-values to them, "
      "and do not use the words wet, drought, or stress detection when describing them.")

In [ ]:
# Item 15: the previous version drew one line per precipitation group, which connected
# non-contiguous years (e.g. 2018 straight to 2021) with a long straight segment that has
# no data behind it. The NDVI trace is now a single continuous series per crop, and the
# precipitation group is shown as a shaded background band for each year instead.
import matplotlib.patches as mpatches

GROUP_COLORS = {PRECIP_GROUP_HIGH: "#cfe3f7", PRECIP_GROUP_LOW: "#f7e0cf"}

def plot_baseline_ndvi(df, precip_table, out_path):
    crops = sorted(df["crop"].unique())
    fig, axes = plt.subplots(len(crops), 1, figsize=(11, 2.6*len(crops)), sharex=True)
    if len(crops) == 1:
        axes = [axes]
    for ax, crop in zip(axes, crops):
        for _, r in precip_table.iterrows():
            ax.axvspan(pd.Timestamp(f"{int(r['year'])}-01-01"),
                       pd.Timestamp(f"{int(r['year'])}-12-31"),
                       color=GROUP_COLORS.get(r["precip_group"], "#eeeeee"),
                       alpha=0.45, lw=0, zorder=0)
        sub = df[df["crop"] == crop].sort_values("date")
        ax.plot(sub["date"], sub["ndvi_mean"], color="#1a1a1a", lw=1.0,
                marker="o", ms=2.2, zorder=2)
        ax.set_ylabel("Mean NDVI"); ax.set_ylim(0, 1)
        ax.text(0.01, 0.86, crop.lower(), transform=ax.transAxes, fontsize=11, fontweight="bold")
        ax.grid(True, alpha=0.2, zorder=1)
    handles = [mpatches.Patch(color=c, alpha=0.45, label=g) for g, c in GROUP_COLORS.items()]
    axes[0].legend(handles=handles, loc="upper right", fontsize=8, ncol=2)
    axes[-1].set_xlabel("Date")
    fig.tight_layout()
    fig.savefig(out_path, dpi=250, bbox_inches="tight"); plt.show()

baseline_fig = FIG_DIR / "figure_baseline_ndvi_by_crop.png"
plot_baseline_ndvi(baseline_df, precip_df, baseline_fig)
print("Saved:", baseline_fig)
print("Caption note: the black trace is the per-scene regional mean NDVI; background shading "
      "marks each year's precipitation group. No line segment spans a data gap.")

Step 4: building SR training pairs

saved as LR/HR arrays (band order [B4, B3, B2, B8]) in data_pairs/sr_pairs_x4/


Caching: if minimum number of pairs (15,000) already exist and REGENERATE_PAIRS=False, generation is skipped.

In [ ]:
# ---- pure-Python helpers for pair building ----
def structured_npy_to_hwc(arr, bands):
    return np.stack([arr[b].astype(np.float32) for b in bands], axis=-1)

def to_reflectance(hr_raw):
    return np.clip(hr_raw.astype(np.float32) / 10000.0, 0.0, 1.0)

def downsample_area_exact(hr, scale):
    H, W, C = hr.shape
    assert H % scale == 0 and W % scale == 0
    return hr.reshape(H//scale, scale, W//scale, scale, C).mean(axis=(1, 3)).astype(np.float32)

def is_valid_patch(hr_raw):
    if not np.isfinite(hr_raw).all():
        return False
    if float((hr_raw == 0).all(axis=-1).mean()) > 0.05:
        return False
    if float(hr_raw.std()) < 1e-6:
        return False
    return True

def pair_filename(year, lon, lat):
    """Deterministic name so the same coordinate always maps to the same file (Item 6)."""
    return f"S2_{year}_{lon:.5f}_{lat:.5f}.npz".replace("-", "m")

def download_hr_patch(image, lon, lat, size_px, scale_m, bands):
    half_m = (size_px * scale_m) / 2.0
    region = ee.Geometry.Point([lon, lat]).buffer(half_m).bounds()
    try:
        url = image.getDownloadURL({"bands": bands, "region": region,
                                    "dimensions": f"{size_px}x{size_px}", "format": "NPY"})
        raw = urllib.request.urlopen(url, timeout=120).read()
        return structured_npy_to_hwc(np.load(io.BytesIO(raw)), bands)
    except Exception as e:
        print("   download failed:", str(e)[:100]); return None

def crop_candidates(year, n, seed):
    """Random points on target-crop pixels, filtered by COLLECTIVE target-crop coverage.

    The MIN_CROP_FRACTION threshold applies to the UNION of all target crop classes within
    the patch window, not to a single crop. The dominant target crop and its own fraction
    are recorded per patch at save time. Returns (lon, lat, coverage, dom_code, dom_frac)."""
    is_crop = cdl_is_crop(year, CROP_CODE_LIST)
    pts = is_crop.stratifiedSample(numPoints=0, classBand="is_crop", region=AOI, scale=CDL_SCALE,
                                   classValues=[1], classPoints=[n], seed=seed, tileScale=16,
                                   geometries=True)
    half_m = (PATCH_SIZE_HR * HR_SCALE_METERS) / 2.0
    cdl_img = get_cdl_image(year)
    cdl_band = cdl_crop_band_name(cdl_img)

    def add_stats(feat):
        win = feat.geometry().buffer(half_m).bounds()
        # collective target-crop coverage
        frac = is_crop.reduceRegion(reducer=ee.Reducer.mean(), geometry=win, scale=CDL_SCALE,
                                    maxPixels=int(1e8), tileScale=16).get("is_crop")
        # full class histogram, so the dominant crop can be resolved without a second pass
        hist = cdl_img.select([cdl_band]).reduceRegion(
            reducer=ee.Reducer.frequencyHistogram(), geometry=win, scale=CDL_SCALE,
            maxPixels=int(1e8), tileScale=16).get(cdl_band)
        return feat.set("crop_frac", frac).set("cdl_hist", hist)

    feats = pts.map(add_stats).getInfo()["features"]
    out = []
    for f in feats:
        props = f["properties"]
        fr = props.get("crop_frac")
        if fr is None or fr < MIN_CROP_FRACTION:
            continue
        hist = props.get("cdl_hist") or {}
        counts = {int(k): v for k, v in hist.items()}
        total = sum(counts.values())
        target_counts = {c: v for c, v in counts.items() if c in CROP_CODE_LIST}
        if target_counts and total:
            dom_code = max(target_counts, key=target_counts.get)
            dom_frac = target_counts[dom_code] / total
        else:
            dom_code, dom_frac = -1, float("nan")
        lon, lat = f["geometry"]["coordinates"][0], f["geometry"]["coordinates"][1]
        out.append((lon, lat, float(fr), int(dom_code), float(dom_frac)))
    return out

def code_to_crop_name(code):
    return next((name for name, c in CROP_CODES.items() if c == code), None)

def save_pair(path, lr, hr, meta):
    """Item 7: write geolocation and crop metadata INTO the .npz at generation time, so
    none of it has to be recomputed later through thousands of Earth Engine requests."""
    np.savez_compressed(
        path,
        lr=lr.astype(np.float32), hr=hr.astype(np.float32),
        year=np.int32(meta["year"]),
        lon=np.float64(meta["lon"]), lat=np.float64(meta["lat"]),
        crop_coverage=np.float32(meta["crop_coverage"]),
        dominant_crop_code=np.int32(meta["dominant_crop_code"]),
        dominant_crop_fraction=np.float32(meta["dominant_crop_fraction"]),
        band_order=np.array(SR_BANDS),
        scale_factor=np.int32(SCALE),
        composite_window=np.array([COMPOSITE_START_MONTHDAY, COMPOSITE_END_MONTHDAY]),
    )

def read_pair_meta(path):
    """Return the metadata stored at generation time, or {} for older metadata-free files."""
    try:
        with np.load(path, allow_pickle=False) as d:
            if "year" not in d.files:
                return {}
            code = int(d["dominant_crop_code"])
            return {
                "year": int(d["year"]), "longitude": float(d["lon"]), "latitude": float(d["lat"]),
                "crop_fraction": float(d["crop_coverage"]),
                "dominant_crop_code": code,
                "crop_label": code_to_crop_name(code),
                "dominant_crop_fraction": float(d["dominant_crop_fraction"]),
                "band_order": ",".join([str(b) for b in d["band_order"]]),
            }
    except Exception:
        return {}

print("Pair-building helpers ready.")

In [ ]:
# Item 3/4/5/6/7: generation is resumable, never auto-downloads, rejects duplicate
# coordinates before counting a save, and stores per-patch metadata in the .npz.
import shutil

if USE_CACHED_PAIRS:
    n_cached = len(list(PAIRS_SOURCE.glob("*.npz")))
    if n_cached == 0:
        raise FileNotFoundError(f"USE_CACHED_PAIRS=True but no .npz files in {PAIRS_SOURCE}.")
    run_manifest["sr_pair_data_source"] = f"user_supplied_cache:{PAIRS_SOURCE} ({n_cached} files)"
    print(f"USE_CACHED_PAIRS=True - using {n_cached} pairs from {PAIRS_SOURCE}. Generation skipped.")
else:
    existing_files = sorted(PAIRS_DIR.glob("*.npz"))
    have_enough = len(existing_files) >= MIN_TOTAL_PAIRS

    if RESET_PAIRS:
        if os.environ.get("CVSR_CONFIRM_RESET", "0") != "1":
            raise RuntimeError(f"RESET_PAIRS=True would delete {len(existing_files)} pairs. "
                               "Set CVSR_CONFIRM_RESET=1 to confirm, or leave RESET_PAIRS=False "
                               "to resume generation instead.")
        for f in existing_files:
            f.unlink()
        existing_files = []
        print("RESET_PAIRS: pair directory cleared.")

    if have_enough and not REGENERATE_PAIRS and not RESET_PAIRS:
        run_manifest["sr_pair_data_source"] = f"cached_real_npz:{PAIRS_DIR} ({len(existing_files)} files)"
        print(f"Found {len(existing_files)} pairs in {PAIRS_DIR} - skipping generation.")
    else:
        if not EE_READY:
            raise RuntimeError("SR pairs must be built from Earth Engine, but EE is not initialized. "
                               "Set EE_PROJECT, or point CVSR_CACHED_PAIRS_DIR at existing pairs.")

        # ---- resume state: what is already on disk, keyed by coordinate ----
        seen_coords_gen = set()
        for f in PAIRS_DIR.glob("*.npz"):
            m = read_pair_meta(f)
            if m:
                seen_coords_gen.add((m["year"], round(m["longitude"], 5), round(m["latitude"], 5)))
        saved_by_year = {}
        for f in PAIRS_DIR.glob("*.npz"):
            try:
                yr = int(f.name.split("_")[1])
                saved_by_year[yr] = saved_by_year.get(yr, 0) + 1
            except Exception:
                pass
        print(f"Resuming: {len(list(PAIRS_DIR.glob('*.npz')))} pairs already on disk "
              f"({saved_by_year}). Existing files are kept, not deleted.")

        n_saved_total = n_dup_rejected = n_resume_skipped = 0

        for year in YEARS:
            print(f"\n=== Year {year} ===")
            composite = seasonal_composite(year)
            saved_for_year = int(saved_by_year.get(year, 0))
            if saved_for_year >= NUM_PATCHES_PER_YEAR:
                print(f"   already have {saved_for_year}/{NUM_PATCHES_PER_YEAR} for {year} - skipping.")
                continue
            attempt_seed_offset = 0
            MAX_CANDIDATES_PER_EE_CALL = 4500
            total_candidates_sampled_for_year = 0

            while saved_for_year < NUM_PATCHES_PER_YEAR and \
                  total_candidates_sampled_for_year < (NUM_PATCHES_PER_YEAR * MAX_ATTEMPTS_FACTOR):

                num_to_request = min(MAX_CANDIDATES_PER_EE_CALL,
                                     (NUM_PATCHES_PER_YEAR * MAX_ATTEMPTS_FACTOR)
                                     - total_candidates_sampled_for_year)
                if num_to_request <= 0:
                    break

                cands_batch = crop_candidates(year, num_to_request,
                                              seed=SEED + year + attempt_seed_offset)
                attempt_seed_offset += 1
                total_candidates_sampled_for_year += len(cands_batch)
                if not cands_batch:
                    print(f"   no more candidates for {year}. Saved {saved_for_year}/{NUM_PATCHES_PER_YEAR}")
                    break
                print(f"   {len(cands_batch)} candidates this batch (total sampled "
                      f"{total_candidates_sampled_for_year})")

                for lon, lat, coverage, dom_code, dom_frac in cands_batch:
                    if saved_for_year >= NUM_PATCHES_PER_YEAR:
                        break

                    # Item 6: reject duplicates BEFORE any download or counter increment
                    coord_key = (year, round(lon, 5), round(lat, 5))
                    if coord_key in seen_coords_gen:
                        n_dup_rejected += 1
                        continue
                    out_path = PAIRS_DIR / pair_filename(year, lon, lat)
                    if out_path.exists():
                        seen_coords_gen.add(coord_key)
                        n_resume_skipped += 1
                        continue

                    hr_raw = download_hr_patch(composite, lon, lat, PATCH_SIZE_HR,
                                               HR_SCALE_METERS, SR_BANDS)
                    if hr_raw is None or hr_raw.shape != (PATCH_SIZE_HR, PATCH_SIZE_HR, len(SR_BANDS)):
                        continue
                    if not is_valid_patch(hr_raw):
                        continue

                    hr = to_reflectance(hr_raw)
                    lr = downsample_area_exact(hr, SCALE)
                    save_pair(out_path, lr, hr, {
                        "year": year, "lon": lon, "lat": lat,
                        "crop_coverage": coverage,
                        "dominant_crop_code": dom_code,
                        "dominant_crop_fraction": dom_frac,
                    })
                    seen_coords_gen.add(coord_key)
                    saved_for_year += 1          # only after a genuinely new file is written
                    n_saved_total += 1
                    if saved_for_year % 100 == 0:
                        print(f"   saved {saved_for_year}/{NUM_PATCHES_PER_YEAR} for {year}")
                    time.sleep(0.05)

            print(f"   -> {saved_for_year} patches for {year} (target {NUM_PATCHES_PER_YEAR})")

        n_on_disk = len(list(PAIRS_DIR.glob("*.npz")))
        print(f"\nNew this run: {n_saved_total} | duplicate coords rejected: {n_dup_rejected} "
              f"| already-present skipped: {n_resume_skipped}")
        print(f"Total pairs on disk: {n_on_disk}")
        if n_on_disk < MIN_TOTAL_PAIRS:
            print(f"WARNING: {n_on_disk} < {MIN_TOTAL_PAIRS}. Re-run this cell to resume "
                  "(existing files are preserved), or lower MIN_CROP_FRACTION.")
        run_manifest["sr_pair_data_source"] = f"earth_engine_real:{S2_SR_ID}"
        run_manifest["pairs_new_this_run"] = n_saved_total
        run_manifest["pairs_duplicate_coords_rejected"] = n_dup_rejected

# Item 4: archiving is opt-in and never fires automatically mid-generation.
def archive_pairs(tag="pairs"):
    """Zip the pair directory on request. Downloads only if AUTO_DOWNLOAD is on in Colab."""
    base = str(PROJECT_ROOT / f"sr_pairs_x4_{tag}")
    shutil.make_archive(base, "zip", PAIRS_DIR)
    print("Archive written:", base + ".zip")
    if AUTO_DOWNLOAD and IN_COLAB:
        colab_files.download(base + ".zip")
    return base + ".zip"

save_manifest()

Step 5: load pairs, audit shapes, detect scale, train/val/test split

In [ ]:
# Optional: unpack a previously archived pair set instead of regenerating.
# Item 2/3: no hard-coded Drive path. Point CVSR_PAIRS_ARCHIVE at a .zip anywhere
# (a Drive path, a local file, a mounted volume) and re-run this cell.
_archive = os.environ.get("CVSR_PAIRS_ARCHIVE", "").strip()
if not _archive:
    print("No CVSR_PAIRS_ARCHIVE set - skipping import. "
          "Pairs will be read from:", PAIRS_SOURCE)
else:
    src = Path(_archive)
    if not src.exists():
        raise FileNotFoundError(f"CVSR_PAIRS_ARCHIVE points at {src}, which does not exist.")
    import zipfile
    before = len(list(PAIRS_DIR.glob("*.npz")))
    with zipfile.ZipFile(src) as zf:
        zf.extractall(PAIRS_DIR)
    after = len(list(PAIRS_DIR.glob("*.npz")))
    print(f"Extracted {src} -> {PAIRS_DIR} ({before} -> {after} pairs)")

In [ ]:
import re

pair_files = sorted(PAIRS_SOURCE.glob("*.npz"))
assert len(pair_files) > 0, f"No .npz pairs in {PAIRS_SOURCE}. Run Step 4 or set CVSR_CACHED_PAIRS_DIR."

# file names are written as f"S2_{year}_{lon:.5f}_{lat:.5f}.npz".replace("-", "m")
FNAME_RE = re.compile(r"^S2_(\d+)_(m?[\d.]+)_(m?[\d.]+)\.npz$")

def parse_pair_filename(fname):
    """Recover (year, lon, lat) from the naming convention; (None, None, None) if unmatched.
    Used only as a fallback for pairs written before metadata was stored in the .npz."""
    m = FNAME_RE.match(fname)
    if not m:
        return None, None, None
    def _num(tok):
        return -float(tok[1:]) if tok.startswith("m") else float(tok)
    return int(m.group(1)), _num(m.group(2)), _num(m.group(3))

def read_pair(path):
    d = np.load(path)
    return d["lr"].astype(np.float32), d["hr"].astype(np.float32)

# --- extended data-infrastructure audit ---
# Checks: unique filenames/coordinates, missing/non-finite values, reflectance range,
# 4-band order plausibility, and LR == 4x area-average of HR within tolerance.
audit_rows, valid_files = [], []
seen_coords = {}
LR_HR_ATOL = 1e-4  # numerical tolerance for LR == 4x area-average(HR)

for p in pair_files:
    fname = p.name
    meta = read_pair_meta(p)
    if meta:
        year, lon, lat = meta["year"], meta["longitude"], meta["latitude"]
    else:
        year, lon, lat = parse_pair_filename(fname)
    row = {"file": fname, "year": year, "longitude": lon, "latitude": lat}
    try:
        lr, hr = read_pair(p)
        lh, lw, lc = lr.shape; hh, hw, hc = hr.shape
        sh, sw = hh / lh, hw / lw
        shape_ok = lr.ndim == 3 and hr.ndim == 3 and abs(sh - sw) < 1e-6 and lc == hc

        finite_ok = bool(np.isfinite(lr).all() and np.isfinite(hr).all())
        n_nonfinite = int((~np.isfinite(hr)).sum() + (~np.isfinite(lr)).sum())

        reflect_ok = bool((hr.min() >= -1e-6) and (hr.max() <= 1.0 + 1e-6) and
                          (lr.min() >= -1e-6) and (lr.max() <= 1.0 + 1e-6))

        bands_ok = (hc == len(SR_BANDS)) and (lc == len(SR_BANDS))
        band_order_plausible = bool(hr[..., NIR_IDX].mean() > hr[..., RED_IDX].mean()) if bands_ok else False

        if finite_ok and shape_ok:
            lr_expected = downsample_area_exact(hr, int(round(sh)))
            lr_match = bool(np.allclose(lr, lr_expected, atol=LR_HR_ATOL))
            lr_max_abs_err = float(np.max(np.abs(lr - lr_expected)))
        else:
            lr_match = False
            lr_max_abs_err = np.nan

        # ITEM 1 (FIXED): lr_match is now part of validity. Previously it was computed and
        # then dropped from this expression, so the manuscript claimed a stricter audit
        # than the code enforced. A patch is valid only if it passes EVERY check.
        valid = (
            shape_ok
            and finite_ok
            and reflect_ok
            and bands_ok
            and lr_match
        )
        row.update({
            "lr_shape": str(lr.shape), "hr_shape": str(hr.shape), "scale_h": sh, "channels": hc,
            "shape_ok": shape_ok, "finite_ok": finite_ok, "n_nonfinite": n_nonfinite,
            "reflectance_ok": reflect_ok, "hr_min": float(hr.min()), "hr_max": float(hr.max()),
            "bands_ok": bands_ok, "band_order_plausible": band_order_plausible,
            "lr_matches_4x_avg_hr": lr_match, "lr_max_abs_err_vs_4x_avg": lr_max_abs_err,
            "has_stored_metadata": bool(meta),
            "valid": bool(valid),
        })
        if valid:
            valid_files.append(fname)
    except Exception as e:
        row.update({"error": str(e), "valid": False})
    audit_rows.append(row)

    if lon is not None and lat is not None:
        seen_coords.setdefault((round(lon, 5), round(lat, 5)), []).append(fname)

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(RESULTS_DIR / "data_audit.csv", index=False)
assert valid_files, "No valid LR/HR pairs found."

n_files = len(pair_files)
n_unique_files = audit_df["file"].nunique()
assert n_unique_files == n_files, f"Duplicate filenames on disk: {n_files - n_unique_files} repeats."

dup_coord_groups = {k: v for k, v in seen_coords.items() if len(v) > 1}
n_unique_coords = len(seen_coords)

band_order_ok_frac = float(audit_df["band_order_plausible"].mean()) if "band_order_plausible" in audit_df else np.nan
lr_match_frac = float(audit_df["lr_matches_4x_avg_hr"].mean()) if "lr_matches_4x_avg_hr" in audit_df else np.nan
reflect_ok_frac = float(audit_df["reflectance_ok"].mean()) if "reflectance_ok" in audit_df else np.nan
total_nonfinite = int(audit_df["n_nonfinite"].sum()) if "n_nonfinite" in audit_df else np.nan
year_counts = audit_df["year"].value_counts(dropna=False).sort_index()

n_failed = int((~audit_df["valid"].astype(bool)).sum())
n_failed_lr = int((~audit_df["lr_matches_4x_avg_hr"].fillna(False).astype(bool)).sum())

DETECTED_SCALE = int(round(audit_df[audit_df["valid"]]["scale_h"].round(6).mode().iloc[0]))
s_lr, s_hr = read_pair(PAIRS_SOURCE / valid_files[0])
IN_CHANNELS = s_hr.shape[-1]; HR_PATCH = s_hr.shape[0]; LR_PATCH = s_lr.shape[0]

run_manifest.update({
    "detected_scale": DETECTED_SCALE, "input_channels": IN_CHANNELS,
    "hr_patch": HR_PATCH, "lr_patch": LR_PATCH, "num_pair_files": n_files,
    "audit_n_unique_filenames": n_unique_files, "audit_n_unique_coordinates": n_unique_coords,
    "audit_n_duplicate_coordinate_groups": len(dup_coord_groups),
    "audit_band_order_plausible_fraction": band_order_ok_frac,
    "audit_lr_matches_4x_avg_hr_fraction": lr_match_frac,
    "audit_reflectance_in_range_fraction": reflect_ok_frac,
    "audit_total_nonfinite_values": total_nonfinite,
    "audit_n_invalid_patches": n_failed,
    "audit_lr_match_enforced_in_validity": True,
    "audit_n_valid_patches": len(valid_files),
})
save_manifest()

print("Detected scale:", DETECTED_SCALE, "| channels:", IN_CHANNELS, "| HR", HR_PATCH, "| LR", LR_PATCH)
print(f"Files on disk: {n_files} | unique filenames: {n_unique_files} | unique coordinates: {n_unique_coords}")
print(f"Duplicate-coordinate groups: {len(dup_coord_groups)}"
      + (f" (e.g. {list(dup_coord_groups.items())[0]})" if dup_coord_groups else ""))
print(f"Non-finite values found: {total_nonfinite}")
print(f"Reflectance in [0,1] for {reflect_ok_frac*100:.1f}% of patches")
print(f"4-band order plausible (mean NIR > mean RED) for {band_order_ok_frac*100:.1f}% of patches")
print(f"LR matches 4x area-average of HR (atol={LR_HR_ATOL}) for {lr_match_frac*100:.1f}% of patches")
print(f"Patches with metadata stored at generation time: "
      f"{int(audit_df['has_stored_metadata'].sum()) if 'has_stored_metadata' in audit_df else 0}/{n_files}")
print("Patch counts by year:")
print(year_counts.to_string())
print(f"\nVALID (all checks incl. lr_match): {len(valid_files)}/{n_files} | failed: {n_failed}")
if n_failed_lr:
    print(f"IMPORTANT: {n_failed_lr} patches fail lr_match and are now excluded from the splits. "
          "Report this count in Methods -- with the old code they were silently included.")
if IN_CHANNELS <= max(RED_IDX, NIR_IDX):
    print("WARNING: not enough channels for NDVI (need NIR at index", NIR_IDX, ").")

Step 5b: generate manifest_details.csv

Builds one row per SR pair file recording: file name, year, longitude, latitude, crop fraction, band order, LR/HR shapes, and (best-effort) a dominant crop label.

Crop fraction/label are recomputed from USDA CDL at the patch location when Earth Engine is available; otherwise they are left as NaN/None rather than guessed.



In [ ]:
MANIFEST_DETAILS_CSV = RESULTS_DIR / "manifest_details.csv"

def crop_fraction_and_label(lon, lat, year, size_px=PATCH_SIZE_HR, scale_m=HR_SCALE_METERS):
    """FALLBACK ONLY (Item 7). Pairs generated by the current code already carry crop
    coverage and the dominant crop inside the .npz, so this Earth Engine lookup runs only
    for legacy metadata-free pairs, and its result is cached in manifest_details.csv."""
    if not EE_READY or lon is None or lat is None or year is None:
        return np.nan, None, np.nan
    try:
        cdl = get_cdl_image(year)
        band = cdl_crop_band_name(cdl)
        half_m = (size_px * scale_m) / 2.0
        win = ee.Geometry.Point([lon, lat]).buffer(half_m).bounds()
        hist = cdl.select([band]).reduceRegion(
            reducer=ee.Reducer.frequencyHistogram(), geometry=win,
            scale=CDL_SCALE, maxPixels=int(1e8), tileScale=8,
        ).get(band).getInfo() or {}
        total = sum(hist.values())
        if total == 0:
            return np.nan, None, np.nan
        code_counts = {int(k): v for k, v in hist.items()}
        crop_total = sum(v for c, v in code_counts.items() if c in CROP_CODE_LIST)
        frac = crop_total / total
        target_counts = {c: v for c, v in code_counts.items() if c in CROP_CODE_LIST}
        label, dom_frac = None, np.nan
        if target_counts:
            dominant_code = max(target_counts, key=target_counts.get)
            label = code_to_crop_name(dominant_code)
            dom_frac = target_counts[dominant_code] / total
        return float(frac), label, float(dom_frac)
    except Exception as e:
        print("   crop-info lookup failed for", lon, lat, year, ":", str(e)[:80])
        return np.nan, None, np.nan

if MANIFEST_DETAILS_CSV.exists() and not REGENERATE_MANIFEST_DETAILS:
    prior_manifest_df = pd.read_csv(MANIFEST_DETAILS_CSV)
else:
    prior_manifest_df = pd.DataFrame(columns=["file"])
prior_lookup = prior_manifest_df.set_index("file").to_dict("index") if len(prior_manifest_df) else {}

manifest_rows = []
n_from_npz = n_from_cache = n_from_ee = 0
for _, arow in audit_df.iterrows():
    fname = arow["file"]
    stored = read_pair_meta(PAIRS_SOURCE / fname)

    if stored:                                   # metadata written at generation time
        n_from_npz += 1
        row = {
            "file": fname, "year": stored["year"],
            "longitude": stored["longitude"], "latitude": stored["latitude"],
            "crop_fraction": stored["crop_fraction"],
            "crop_label": stored["crop_label"],
            "dominant_crop_fraction": stored["dominant_crop_fraction"],
            "band_order": stored["band_order"],
            "metadata_source": "npz_at_generation",
        }
    elif fname in prior_lookup and not REGENERATE_MANIFEST_DETAILS:
        n_from_cache += 1
        row = dict(prior_lookup[fname]); row["file"] = fname
        row.setdefault("metadata_source", "cached_csv")
    else:                                        # legacy pair: one EE lookup, then cached
        n_from_ee += 1
        year, lon, lat = parse_pair_filename(fname)
        frac, label, dom_frac = crop_fraction_and_label(lon, lat, year)
        row = {
            "file": fname, "year": year, "longitude": lon, "latitude": lat,
            "crop_fraction": frac, "crop_label": label, "dominant_crop_fraction": dom_frac,
            "band_order": ",".join(SR_BANDS), "metadata_source": "earth_engine_lookup",
        }
    row["lr_shape"] = arow.get("lr_shape")
    row["hr_shape"] = arow.get("hr_shape")
    row["valid"] = arow.get("valid")
    manifest_rows.append(row)

manifest_details_df = pd.DataFrame(manifest_rows)
manifest_details_df.to_csv(MANIFEST_DETAILS_CSV, index=False)
run_manifest["manifest_details_csv"] = str(MANIFEST_DETAILS_CSV)
run_manifest["manifest_details_rows"] = len(manifest_details_df)
run_manifest["manifest_metadata_from_npz"] = n_from_npz
run_manifest["manifest_metadata_from_ee_lookup"] = n_from_ee
save_manifest()

display(manifest_details_df.head())
print("Saved per-patch manifest:", MANIFEST_DETAILS_CSV, "| rows:", len(manifest_details_df))
print(f"Metadata source - stored in .npz: {n_from_npz} | cached CSV: {n_from_cache} | "
      f"Earth Engine lookup: {n_from_ee}")
if n_from_ee:
    print("NOTE: those EE lookups are cached in manifest_details.csv and will not repeat. "
          "Pairs generated by the current Step 4 carry their metadata already.")
if manifest_details_df["crop_label"].isna().all():
    print("NOTE: crop_label is entirely empty - Earth Engine was unavailable and the pairs "
          "predate stored metadata. Re-run with EE_PROJECT set to populate it.")

Before committing to a chronological (year-based) train/validation/test split, we check whether every target crop is adequately represented in the candidate test year (2022). If a crop is nearly absent in 2022, the chronological split would make its test metric meaningless, so we fall back to a grouped spatial-block split instead -- never an unrestricted random per-patch split.

In [ ]:
# crop_fraction >= MIN_CROP_FRACTION check
low_frac = manifest_details_df[manifest_details_df["crop_fraction"].notna() &
                               (manifest_details_df["crop_fraction"] < MIN_CROP_FRACTION)]
print(f"Patches below the crop-fraction floor ({MIN_CROP_FRACTION}): {len(low_frac)} / {len(manifest_details_df)}")

# crop / year counts (across the WHOLE dataset, not just the eventual test split)
year_crop_counts = (manifest_details_df.groupby(["year", "crop_label"], dropna=False)
                    .size().reset_index(name="count"))
year_crop_counts.to_csv(RESULTS_DIR / "patch_counts_by_year_crop.csv", index=False)
pivot_year_crop = year_crop_counts.pivot(index="year", columns="crop_label", values="count").fillna(0).astype(int)
display(pivot_year_crop)

crop_totals = manifest_details_df["crop_label"].value_counts(dropna=False)
labeled_totals = manifest_details_df["crop_label"].value_counts(dropna=True)
dominant_crop = labeled_totals.idxmax() if len(labeled_totals) else "unlabeled"
dominant_share = float(labeled_totals.max() / labeled_totals.sum()) if len(labeled_totals) else float("nan")
print("Total patches by crop:\n", crop_totals.to_string())
if dominant_share == dominant_share and dominant_share > 0.5:
    print(f"NOTE: the dataset is dominated by {dominant_crop} ({dominant_share*100:.1f}% of all "
          "labelled patches). Do not describe the SR dataset as balanced across crops.")

# ---- Explicit presence/absence check across the WHOLE SR dataset -------------------
# A target crop with ZERO patches is ABSENT, not "adequately represented". The previous
# version skipped zero-count crops (its loop was guarded by `total_for_crop > 0`) and then
# fell through to an "every target crop has adequate representation" message, which is
# false whenever a crop is missing entirely.
crop_patch_counts = {c: int(labeled_totals.get(c, 0)) for c in TARGET_CROPS}
sr_absent_crops_dataset = [c for c in TARGET_CROPS if crop_patch_counts[c] == 0]
sr_present_crops        = [c for c in TARGET_CROPS if crop_patch_counts[c] > 0]

print("\nTarget-crop coverage across all", len(manifest_details_df), "SR patches:")
for c in TARGET_CROPS:
    n = crop_patch_counts[c]
    flag = "  <-- ABSENT: no SR patches, no SR claim possible" if n == 0 else ""
    print(f"  {c:<8}: {n:>6} patches{flag}")

if sr_absent_crops_dataset:
    print(f"\nABSENT FROM THE SR DATASET ENTIRELY: {sr_absent_crops_dataset}")
    print("  These crops have zero patches, so the SR model was never trained or evaluated")
    print("  on them. They cannot appear in any per-crop SR result, and the aggregate SR")
    print("  metrics do not represent them. They remain valid in the baseline NDVI analysis,")
    print("  which is computed from CDL-masked scene means and does not use patches.")
    undeclared = [c for c in sr_absent_crops_dataset if c not in SR_CROPS_EXCLUDED_FROM_CLAIMS]
    if undeclared:
        print(f"  WARNING: {undeclared} are absent but NOT listed in "
              "SR_CROPS_EXCLUDED_FROM_CLAIMS. Add them there and remove them from every SR "
              "claim in the manuscript.")

# ---- Split-strategy decision, evaluated only over crops that actually exist ---------
counts_test_year = manifest_details_df[
    manifest_details_df["year"] == SPLIT_TEST_YEAR]["crop_label"].value_counts(dropna=False)
print(f"\nCrop counts in candidate test year {SPLIT_TEST_YEAR}:\n", counts_test_year.to_string())

underrepresented = []
for crop_name in sr_present_crops:
    total_for_crop = crop_patch_counts[crop_name]
    test_year_for_crop = int(counts_test_year.get(crop_name, 0))
    share = (test_year_for_crop / total_for_crop) if total_for_crop else 0.0
    if test_year_for_crop < MIN_TEST_CROP_COUNT and share < MIN_TEST_CROP_SHARE:
        underrepresented.append(crop_name)
    print(f"  {crop_name}: {test_year_for_crop} in {SPLIT_TEST_YEAR} of {total_for_crop} total "
          f"({share*100:.1f}%)")
for crop_name in sr_absent_crops_dataset:
    print(f"  {crop_name}: absent from the dataset - excluded from the split decision "
          "(a spatial-block fallback cannot conjure patches that do not exist)")

if underrepresented:
    SPLIT_STRATEGY = "spatial_block"
    print(f"\nDECISION: {underrepresented} are too thin in {SPLIT_TEST_YEAR}. "
          "Falling back to a grouped SPATIAL-BLOCK split instead of the chronological split.")
else:
    SPLIT_STRATEGY = "temporal"
    print(f"\nDECISION: every target crop THAT IS PRESENT IN THE DATASET ({sr_present_crops}) "
          f"has adequate representation in {SPLIT_TEST_YEAR}. Using the CHRONOLOGICAL split: "
          f"train={SPLIT_TRAIN_YEARS}, val={SPLIT_VAL_YEAR}, test={SPLIT_TEST_YEAR}.")
    if sr_absent_crops_dataset:
        print(f"         This statement EXCLUDES {sr_absent_crops_dataset}, which are absent "
              "from the SR dataset entirely.")

run_manifest["split_strategy"] = SPLIT_STRATEGY
run_manifest["split_underrepresented_crops_in_test_year"] = underrepresented
run_manifest["sr_target_crops"] = TARGET_CROPS
run_manifest["sr_crop_patch_counts"] = crop_patch_counts
run_manifest["sr_crops_present_in_dataset"] = sr_present_crops
run_manifest["sr_crops_absent_from_dataset"] = sr_absent_crops_dataset
run_manifest["sr_crops_excluded_from_claims"] = list(SR_CROPS_EXCLUDED_FROM_CLAIMS)
save_manifest()

# ---- dataset_summary.csv: crop counts (including explicit zeros), years, fraction stats ----
frac_stats = manifest_details_df["crop_fraction"].describe()
summary_lines = []
for crop_name in TARGET_CROPS:                      # every target crop, zeros included
    summary_lines.append({"group_type": "crop", "group": crop_name,
                          "value": crop_patch_counts[crop_name]})
n_unlabeled = int(manifest_details_df["crop_label"].isna().sum())
if n_unlabeled:
    summary_lines.append({"group_type": "crop", "group": "unlabeled", "value": n_unlabeled})
for yr, cnt in manifest_details_df["year"].value_counts().sort_index().items():
    summary_lines.append({"group_type": "year", "group": int(yr), "value": int(cnt)})
for stat_name, key in [("mean", "mean"), ("std", "std"), ("min", "min"), ("median", "50%"), ("max", "max")]:
    summary_lines.append({"group_type": "crop_fraction_stat", "group": stat_name,
                          "value": float(frac_stats.get(key, np.nan))})
summary_lines.append({"group_type": "total", "group": "all_patches", "value": int(len(manifest_details_df))})
dataset_summary_df = pd.DataFrame(summary_lines)
dataset_summary_df.to_csv(RESULTS_DIR / "dataset_summary.csv", index=False)
display(dataset_summary_df)
print("Saved:", RESULTS_DIR / "dataset_summary.csv", "|", RESULTS_DIR / "patch_counts_by_year_crop.csv")

build split

In [ ]:
manifest_lookup_full = manifest_details_df.set_index("file").to_dict("index")

if SPLIT_STRATEGY == "temporal":
    def _split_of(fname):
        y = manifest_lookup_full.get(fname, {}).get("year")
        if y in SPLIT_TRAIN_YEARS: return "train"
        if y == SPLIT_VAL_YEAR:    return "val"
        if y == SPLIT_TEST_YEAR:   return "test"
        return None
    split_assignment = {f: _split_of(f) for f in valid_files}

else:  # spatial_block
    def _block_id(lon, lat, size=GEO_BLOCK_SIZE_DEG):
        return (math.floor(lon / size), math.floor(lat / size))
    blocks = {}
    for f in valid_files:
        meta = manifest_lookup_full.get(f, {})
        lon, lat = meta.get("longitude"), meta.get("latitude")
        if lon is None or lat is None:
            continue
        blocks.setdefault(_block_id(lon, lat), []).append(f)
    block_ids = sorted(blocks.keys())
    rng_blocks = np.random.default_rng(SEED)
    rng_blocks.shuffle(block_ids)
    n_blocks = len(block_ids)
    n_train_blk = int(round(SPATIAL_SPLIT_FRACTIONS["train"] * n_blocks))
    n_val_blk   = int(round(SPATIAL_SPLIT_FRACTIONS["val"]   * n_blocks))
    train_blocks = set(block_ids[:n_train_blk])
    val_blocks   = set(block_ids[n_train_blk:n_train_blk + n_val_blk])
    test_blocks  = set(block_ids[n_train_blk + n_val_blk:])
    split_assignment = {}
    for bid, files_in_block in blocks.items():
        s = "train" if bid in train_blocks else ("val" if bid in val_blocks else "test")
        for f in files_in_block:
            split_assignment[f] = s
    print(f"Spatial blocks: {n_blocks} total (~{GEO_BLOCK_SIZE_DEG} deg each) "
          f"-> {len(train_blocks)} train / {len(val_blocks)} val / {len(test_blocks)} test blocks")

train_names_all = sorted(f for f, s in split_assignment.items() if s == "train")
val_names_all   = sorted(f for f, s in split_assignment.items() if s == "val")
test_names_all  = sorted(f for f, s in split_assignment.items() if s == "test")

assert (len(train_names_all) + len(val_names_all) + len(test_names_all) ==
        sum(1 for s in split_assignment.values() if s is not None)), \
    "Some patches were not assigned a split."

def _split_rows(names):
    out = []
    for f in names:
        meta = manifest_lookup_full.get(f, {})
        out.append({"file": f, "year": meta.get("year"), "longitude": meta.get("longitude"),
                    "latitude": meta.get("latitude"), "crop_label": meta.get("crop_label")})
    return pd.DataFrame(out)

train_split_df = _split_rows(train_names_all)
val_split_df   = _split_rows(val_names_all)
test_split_df  = _split_rows(test_names_all)
train_split_df.to_csv(TRAIN_CSV, index=False)
val_split_df.to_csv(VAL_CSV, index=False)
test_split_df.to_csv(TEST_CSV, index=False)

# no overlap between train / validation / test filenames
s_train, s_val, s_test = set(train_names_all), set(val_names_all), set(test_names_all)
assert not (s_train & s_val), "train/val overlap!"
assert not (s_train & s_test), "train/test overlap!"
assert not (s_val & s_test), "val/test overlap!"

run_manifest.update({
    "split_strategy_used": SPLIT_STRATEGY,
    "num_train_files": len(train_names_all), "num_val_files": len(val_names_all),
    "num_test_files": len(test_names_all),
})
save_manifest()

print(f"Split strategy: {SPLIT_STRATEGY}")
print(f"Train: {len(train_names_all)} | Val: {len(val_names_all)} | Test: {len(test_names_all)}")
print("No overlap between train/val/test filenames: confirmed.")
print("Saved:", TRAIN_CSV, "|", VAL_CSV, "|", TEST_CSV)

# diagnostic map of the split (helps visually confirm no leakage across nearby patches)
try:
    plt.figure(figsize=(6, 5))
    for name, df_, color in [("train", train_split_df, "tab:blue"),
                             ("val", val_split_df, "tab:orange"),
                             ("test", test_split_df, "tab:green")]:
        plt.scatter(df_["longitude"], df_["latitude"], s=3, alpha=0.4, label=name, color=color)
    plt.xlabel("Longitude"); plt.ylabel("Latitude"); plt.legend(markerscale=4)
    plt.title(f"Patch split map ({SPLIT_STRATEGY})")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "figure_split_map.png", dpi=200, bbox_inches="tight")
    plt.show()
except Exception as e:
    print("Split map skipped:", e)


In [ ]:
class SRPairsDataset(Dataset):
    def __init__(self, pair_dir, file_names, max_items=None):
        self.pair_dir = Path(pair_dir)
        self.files = list(file_names)[:max_items] if max_items else list(file_names)
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        lr, hr = read_pair(self.pair_dir / self.files[idx])
        if hr.max() > 2.0 or lr.max() > 2.0:      # safety: rescale if stored as raw DN
            hr = hr / 10000.0; lr = lr / 10000.0
        hr = np.clip(hr, 0, 1).astype(np.float32)
        lr = np.clip(lr, 0, 1).astype(np.float32)
        return (torch.from_numpy(np.transpose(lr, (2, 0, 1))),
                torch.from_numpy(np.transpose(hr, (2, 0, 1))))

MAX_TRAIN = None
MAX_VAL   = None
BATCH_SIZE = 16 if str(device) == "cuda" else 4

train_ds = SRPairsDataset(PAIRS_SOURCE, train_names_all, max_items=MAX_TRAIN)
val_ds   = SRPairsDataset(PAIRS_SOURCE, val_names_all,   max_items=MAX_VAL)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
# The test set is deliberately NOT wrapped in a batched DataLoader: every test patch is
# scored individually in Step 7 so the reported metrics are batch-size independent.
run_manifest["num_train_patches_used"] = len(train_ds)
run_manifest["num_val_patches_used"] = len(val_ds)
run_manifest["num_test_patches_used"] = len(test_names_all)
run_manifest["batch_size"] = BATCH_SIZE
save_manifest()

lr_b, hr_b = next(iter(train_loader))
print("Train:", len(train_ds), "| Val:", len(val_ds), "| Test:", len(test_names_all))
print("LR batch:", tuple(lr_b.shape), "| HR batch:", tuple(hr_b.shape))
print("Tensors are channels-first (N, C, H, W) - describe them that way in Methods.")

In [ ]:
# data sanity figure (LR vs bicubic-upsampled vs HR), using the same fixed display stretch
# as every other RGB figure in the paper (Item 16).
def chw_to_hwc(x):
    if x.ndim == 4:
        x = x[0]
    return np.transpose(x.detach().cpu().numpy(), (1, 2, 0))

def to_display_rgb(img_hwc, stretch=DISPLAY_STRETCH):
    """Fixed reflectance stretch for display ONLY. Band order is B4,B3,B2,B8, so the RGB
    panel is channels [0,1,2]. This never touches the arrays used for metrics."""
    if img_hwc.shape[-1] < 3:
        img_hwc = np.repeat(img_hwc[..., :1], 3, axis=-1)
    rgb = img_hwc[..., :3]
    lo, hi = stretch
    return np.clip((rgb - lo) / (hi - lo), 0, 1)

rgb_like = to_display_rgb   # backwards-compatible alias

lr0, hr0 = lr_b[0:1], hr_b[0:1]
lr0_up = F.interpolate(lr0, size=hr0.shape[-2:], mode="bicubic", align_corners=False)
plt.figure(figsize=(10, 3))
for i, (title, ten) in enumerate([("LR input", lr0), ("Bicubic LR->HR", lr0_up), ("HR target", hr0)], 1):
    plt.subplot(1, 3, i); plt.imshow(to_display_rgb(chw_to_hwc(ten))); plt.title(title); plt.axis("off")
plt.suptitle(f"Display stretch {DISPLAY_STRETCH} reflectance (visualisation only)", fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_data_sanity_patch.png", dpi=200, bbox_inches="tight"); plt.show()

Step 6: Scale-are SR model, metrics, training

Residual CNN that bicubic-upsamples the LR input by the detected scale, then learns a residual correction.

Image reconstruction Metrics: (PSNR, MAE, MSE, SSIM)
Vegetation signal metric: NDVI RMSE

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1)
    def forward(self, x):
        return F.relu(x + self.conv2(F.relu(self.conv1(x))))

class ResidualSRNet(nn.Module):
    def __init__(self, in_channels, hidden_channels=64, num_blocks=6, scale=2):
        super().__init__()
        self.scale = scale
        self.entry = nn.Conv2d(in_channels, hidden_channels, 5, padding=2)
        self.blocks = nn.Sequential(*[ResidualBlock(hidden_channels) for _ in range(num_blocks)])
        self.exit = nn.Conv2d(hidden_channels, in_channels, 3, padding=1)
    def forward(self, x):
        x_up = F.interpolate(x, scale_factor=self.scale, mode="bicubic", align_corners=False)
        h = F.relu(self.entry(x_up))
        h = self.blocks(h)
        return torch.clamp(x_up + self.exit(h), 0, 1)

model = ResidualSRNet(in_channels=IN_CHANNELS, hidden_channels=64, num_blocks=6, scale=DETECTED_SCALE).to(device)
print(model)

In [ ]:
def mae_t(a, b):  return F.l1_loss(a, b).item()
def mse_t(a, b):  return F.mse_loss(a, b).item()
def psnr_t(a, b):
    m = F.mse_loss(a, b).item()
    return float("inf") if m == 0 else -10 * math.log10(m)
def grad_loss(p, t):
    return (F.l1_loss(p[:, :, :, 1:] - p[:, :, :, :-1], t[:, :, :, 1:] - t[:, :, :, :-1]) +
            F.l1_loss(p[:, :, 1:, :] - p[:, :, :-1, :], t[:, :, 1:, :] - t[:, :, :-1, :]))
def sr_loss(p, t):
    return F.l1_loss(p, t) + 0.05 * grad_loss(p, t)

def compute_ndvi_tensor(x, red_idx=RED_IDX, nir_idx=NIR_IDX, eps=1e-6):
    if x.shape[1] <= max(red_idx, nir_idx):
        return None
    red = x[:, red_idx:red_idx+1]; nir = x[:, nir_idx:nir_idx+1]
    return torch.clamp((nir - red) / (nir + red + eps), -1, 1)
def ndvi_rmse_t(a, b):
    na, nb = compute_ndvi_tensor(a), compute_ndvi_tensor(b)
    if na is None or nb is None:
        return np.nan
    return torch.sqrt(F.mse_loss(na, nb)).item()

def ssim_metric(a, b):
    if not SKIMAGE_AVAILABLE:
        return np.nan
    A = a.detach().cpu().numpy(); B = b.detach().cpu().numpy()
    vals = []
    for i in range(A.shape[0]):
        ah = np.transpose(A[i], (1, 2, 0)); bh = np.transpose(B[i], (1, 2, 0))
        vals.append(ssim(ah, bh, data_range=1.0, channel_axis=-1))
    return float(np.mean(vals))

# quick correctness self-check of the NDVI band order
_t = torch.zeros(1, IN_CHANNELS, 4, 4); _t[:, RED_IDX] = 0.1; _t[:, NIR_IDX] = 0.5
_nd = compute_ndvi_tensor(_t)
print("NDVI band-order self-check:", round(float(_nd.mean()), 4), "(expected ~0.6667 for veg)")
print("Metric helpers ready.")

In [ ]:
# NOTE: this is a coarse, BATCH-AVERAGED sanity check on the VALIDATION set only
def evaluate_bicubic(loader):
    rows = []
    for lr, hr in loader:
        lr = lr.to(device); hr = hr.to(device)
        bic = torch.clamp(F.interpolate(lr, size=hr.shape[-2:], mode="bicubic", align_corners=False), 0, 1)
        rows.append({"mae": mae_t(bic, hr), "mse": mse_t(bic, hr), "psnr": psnr_t(bic, hr),
                     "ndvi_rmse": ndvi_rmse_t(bic, hr), "ssim": ssim_metric(bic, hr)})
    return pd.DataFrame(rows).mean(numeric_only=True).to_dict()

bicubic_metrics = evaluate_bicubic(val_loader)
print("Bicubic baseline (validation, batch-mean, diagnostic only):",
      {k: round(v, 5) for k, v in bicubic_metrics.items()})


In [ ]:
LEARNING_RATE = 5e-4
NUM_EPOCHS    = 30
PATIENCE      = 8

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
best_val_loss = float("inf"); best_epoch = -1; epochs_no_improve = 0
history = []
best_model_path = CHECKPOINT_DIR / "best_sr_model.pt"

# NOTE: also a batch-averaged, VALIDATION-set-only diagnostic, used only to pick the
# best checkpoint (early stopping). The final reported TEST metrics are computed
# per-patch in Step 7, not from this function.
def evaluate_model():
    model.eval(); rows = []
    with torch.no_grad():
        for lr, hr in val_loader:
            lr = lr.to(device); hr = hr.to(device); pred = model(lr)
            rows.append({"loss": sr_loss(pred, hr).item(), "mae": mae_t(pred, hr), "mse": mse_t(pred, hr),
                         "psnr": psnr_t(pred, hr), "ndvi_rmse": ndvi_rmse_t(pred, hr),
                         "ssim": ssim_metric(pred, hr)})
    return pd.DataFrame(rows).mean(numeric_only=True).to_dict()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train(); losses = []
    for lr, hr in train_loader:
        lr = lr.to(device); hr = hr.to(device)
        optimizer.zero_grad()
        loss = sr_loss(model(lr), hr)
        loss.backward(); optimizer.step()
        losses.append(loss.item())
    train_loss = float(np.mean(losses))
    vm = evaluate_model()
    history.append({"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in vm.items()}})
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | train_loss={train_loss:.5f} | "
          f"val_loss={vm['loss']:.5f} | val_psnr={vm['psnr']:.3f}")
    if vm["loss"] < best_val_loss:
        best_val_loss = vm["loss"]; best_epoch = epoch; epochs_no_improve = 0
        torch.save({"model_state_dict": model.state_dict(), "scale": DETECTED_SCALE,
                    "in_channels": IN_CHANNELS, "epoch": epoch, "val_metrics": vm}, best_model_path)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print("Early stopping."); break

history_df = pd.DataFrame(history)
history_df.to_csv(RESULTS_DIR / "sr_training_history.csv", index=False)
display(history_df.tail())
print("Best epoch:", best_epoch, "| saved:", best_model_path)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="Train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="Validation")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.25)
axes[1].plot(history_df["epoch"], history_df["val_psnr"], label="Validation PSNR")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("PSNR (dB)"); axes[1].legend(); axes[1].grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "figure_sr_training_curves.png", dpi=250, bbox_inches="tight"); plt.show()

Step 7: final evaluation

Outputs: test_patch_metrics.csv (one row per test patch), sr_bicubic_vs_model_metrics.csv (mean/std/median/95% CI per metric per method, plus the paired model-minus-bicubic difference and a paired t-test p-value), and sr_metrics_by_crop.csv (the same comparison broken out by crop, so crop-specific claims are backed by crop-specific test evidence).

In [ ]:
# reload best checkpoint (weights_only=False so the metadata dict loads on new PyTorch)
checkpoint = torch.load(best_model_path, map_location="cpu", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device); model.eval()
print("Reloaded best model from epoch", checkpoint.get("epoch"))

manifest_lookup = manifest_details_df.set_index("file").to_dict("index")

def eval_one_test_patch(fname, keep_tensors=False):
    """Score exactly ONE patch: bicubic vs model, both against its own HR target.
    No batching and no averaging here -- aggregation happens afterwards over the per-patch
    rows, which is what makes the reported numbers batch-size independent."""
    lr, hr = read_pair(PAIRS_SOURCE / fname)
    if hr.max() > 2.0 or lr.max() > 2.0:
        hr = hr / 10000.0; lr = lr / 10000.0
    hr = np.clip(hr, 0, 1).astype(np.float32); lr = np.clip(lr, 0, 1).astype(np.float32)
    lr_t = torch.from_numpy(np.transpose(lr, (2, 0, 1))).unsqueeze(0).to(device)
    hr_t = torch.from_numpy(np.transpose(hr, (2, 0, 1))).unsqueeze(0).to(device)
    with torch.no_grad():
        bic = torch.clamp(F.interpolate(lr_t, size=hr_t.shape[-2:], mode="bicubic",
                                        align_corners=False), 0, 1)
        pred = model(lr_t)
    meta = manifest_lookup.get(fname, {})
    row = {
        "file": fname, "year": meta.get("year"), "crop_label": meta.get("crop_label"),
        "crop_fraction": meta.get("crop_fraction"),
        "bicubic_mae": mae_t(bic, hr_t), "model_mae": mae_t(pred, hr_t),
        "bicubic_mse": mse_t(bic, hr_t), "model_mse": mse_t(pred, hr_t),
        "bicubic_psnr": psnr_t(bic, hr_t), "model_psnr": psnr_t(pred, hr_t),
        "bicubic_ssim": ssim_metric(bic, hr_t), "model_ssim": ssim_metric(pred, hr_t),
        "bicubic_ndvi_rmse": ndvi_rmse_t(bic, hr_t), "model_ndvi_rmse": ndvi_rmse_t(pred, hr_t),
    }
    tensors = (lr_t.cpu(), bic.cpu(), pred.cpu(), hr_t.cpu()) if keep_tensors else None
    return row, tensors

# --- pass 1: score every held-out test patch ---
test_patch_rows = []
for fname in test_names_all:
    row, _ = eval_one_test_patch(fname)
    test_patch_rows.append(row)

test_patch_metrics_df = pd.DataFrame(test_patch_rows)
test_patch_metrics_df.to_csv(RESULTS_DIR / "test_patch_metrics.csv", index=False)
print("Saved per-patch TEST metrics:", RESULTS_DIR / "test_patch_metrics.csv",
      "| rows:", len(test_patch_metrics_df))

# --- Item 19: choose qualitative examples by a stated, reproducible rule ---
# The old code kept whichever three patches happened to come first in the loop, which is
# neither reproducible nor defensible against a cherry-picking question.
def select_example_files(df, rule=EXAMPLE_SELECTION_RULE, k=N_FIGURE_EXAMPLES):
    d = df.dropna(subset=["model_psnr"]).sort_values("model_psnr").reset_index(drop=True)
    if len(d) == 0:
        return []
    if rule == "median_psnr":
        mid = len(d) // 2
        start = max(0, min(mid - k // 2, len(d) - k))
        chosen = d.iloc[start:start + k]
    elif rule == "quartiles":
        qs = [0.25, 0.50, 0.75][:k]
        chosen = d.iloc[[int(q * (len(d) - 1)) for q in qs]]
    elif rule == "seeded_random":
        chosen = d.sample(min(k, len(d)), random_state=SEED)
    else:
        raise ValueError(f"Unknown EXAMPLE_SELECTION_RULE: {rule}")
    return chosen

example_rows = select_example_files(test_patch_metrics_df)
example_files = list(example_rows["file"])
example_rows.assign(selection_rule=EXAMPLE_SELECTION_RULE).to_csv(
    RESULTS_DIR / "figure_example_patches.csv", index=False)

test_examples = []
for fname in example_files:
    _, tensors = eval_one_test_patch(fname, keep_tensors=True)
    lr_t, bic_t, pred_t, hr_t = tensors
    ex_psnr = float(test_patch_metrics_df.loc[
        test_patch_metrics_df["file"] == fname, "model_psnr"].iloc[0])
    test_examples.append({"file": fname, "model_psnr": ex_psnr,
                          "lr": lr_t[0], "bicubic": bic_t[0], "model": pred_t[0], "hr": hr_t[0]})

run_manifest["figure_example_selection_rule"] = EXAMPLE_SELECTION_RULE
run_manifest["figure_example_files"] = example_files
save_manifest()

print(f"\nQualitative examples selected by rule '{EXAMPLE_SELECTION_RULE}' "
      f"(recorded in results/figure_example_patches.csv):")
for ex in test_examples:
    print(f"   {ex['file']}  model PSNR {ex['model_psnr']:.2f} dB")
print("State this selection rule in the figure caption.")
display(test_patch_metrics_df.head())

In [ ]:
def mean_ci95(x):
    """Mean, std, median, and a normal-approximation 95% CI, ignoring non-finite values."""
    x = np.asarray(pd.to_numeric(x, errors="coerce"), dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n == 0:
        return dict(n=0, mean=np.nan, std=np.nan, median=np.nan, ci95_low=np.nan, ci95_high=np.nan)
    m = float(x.mean()); s = float(x.std(ddof=1)) if n > 1 else 0.0
    se = s / math.sqrt(n) if n > 0 else np.nan
    return dict(n=n, mean=m, std=s, median=float(np.median(x)),
               ci95_low=m - 1.96 * se, ci95_high=m + 1.96 * se)

METRIC_NAMES = ["mae", "mse", "psnr", "ssim", "ndvi_rmse"]
LOWER_IS_BETTER = {"mae", "mse", "ndvi_rmse"}

summary_rows = []
for metric in METRIC_NAMES:
    for method in ["bicubic", "model"]:
        stats = mean_ci95(test_patch_metrics_df[f"{method}_{metric}"])
        stats.update({"metric": metric, "method": method})
        summary_rows.append(stats)
    # paired comparison: bicubic and model are scored on the SAME patches, so compare the
    # per-patch differences directly rather than two independent-sample summaries
    diff = test_patch_metrics_df[f"model_{metric}"] - test_patch_metrics_df[f"bicubic_{metric}"]
    dstats = mean_ci95(diff)
    dstats.update({"metric": metric, "method": "model_minus_bicubic_paired"})
    if SCIPY_AVAILABLE:
        finite_idx = diff[np.isfinite(diff)].index
        if len(finite_idx) > 1:
            _, pval = scipy_stats.ttest_rel(
                test_patch_metrics_df.loc[finite_idx, f"model_{metric}"],
                test_patch_metrics_df.loc[finite_idx, f"bicubic_{metric}"])
            dstats["paired_p_value"] = float(pval)
    summary_rows.append(dstats)

test_metrics_summary_df = pd.DataFrame(summary_rows)
front_cols = ["metric", "method", "n", "mean", "std", "median", "ci95_low", "ci95_high"]
other_cols = [c for c in test_metrics_summary_df.columns if c not in front_cols]
test_metrics_summary_df = test_metrics_summary_df[front_cols + other_cols]
test_metrics_summary_df.to_csv(RESULTS_DIR / "sr_bicubic_vs_model_metrics.csv", index=False)
display(test_metrics_summary_df.round(5))

def _mean_of(metric, method):
    r = test_metrics_summary_df[(test_metrics_summary_df["metric"] == metric) &
                                (test_metrics_summary_df["method"] == method)]
    return float(r["mean"].iloc[0]) if len(r) else float("nan")

psnr_b, psnr_m = _mean_of("psnr", "bicubic"), _mean_of("psnr", "model")
ndvi_b, ndvi_m = _mean_of("ndvi_rmse", "bicubic"), _mean_of("ndvi_rmse", "model")
print(f"\nTest-set mean PSNR - bicubic: {psnr_b:.3f} dB | model: {psnr_m:.3f} dB "
      f"(n={len(test_patch_metrics_df)} patches)")
print(f"Test-set mean NDVI RMSE - bicubic: {ndvi_b:.4f} | model: {ndvi_m:.4f}")
if psnr_m <= psnr_b:
    print("GUARDRAIL: model did NOT beat bicubic on mean test PSNR. "
          "Report it as a baseline, do not claim SR improvement.")


In [ ]:
# Per-crop SR metrics. Every TARGET crop gets a row, including crops with zero test
# patches -- a groupby alone cannot emit a row for a crop that has no data, which is how
# citrus previously vanished from this table and from the final summary.
observed = {c: sub for c, sub in test_patch_metrics_df.groupby("crop_label", dropna=True)}
unlabeled_sub = test_patch_metrics_df[test_patch_metrics_df["crop_label"].isna()]

def _crop_row(name, sub):
    n = int(len(sub))
    if n == 0:
        status = ("absent - no test patches; SR performance CANNOT be claimed"
                  if crop_patch_counts.get(name, 0) == 0
                  else "absent from test split despite patches elsewhere - investigate")
        return {"crop": name, "dataset_patches": int(crop_patch_counts.get(name, 0)),
                "test_patches": 0,
                "bicubic_psnr": np.nan, "model_psnr": np.nan,
                "bicubic_ndvi_rmse": np.nan, "model_ndvi_rmse": np.nan,
                "claim_status": status}
    status = ("reportable" if n >= MIN_TEST_CROP_COUNT
              else f"insufficient - fewer than {MIN_TEST_CROP_COUNT} test patches")
    return {"crop": name, "dataset_patches": int(crop_patch_counts.get(name, len(sub))),
            "test_patches": n,
            "bicubic_psnr": float(sub["bicubic_psnr"].mean()),
            "model_psnr": float(sub["model_psnr"].mean()),
            "bicubic_ndvi_rmse": float(sub["bicubic_ndvi_rmse"].mean()),
            "model_ndvi_rmse": float(sub["model_ndvi_rmse"].mean()),
            "claim_status": status}

crop_rows = [_crop_row(c, observed.get(c, test_patch_metrics_df.iloc[0:0])) for c in TARGET_CROPS]
if len(unlabeled_sub):
    row = _crop_row("unlabeled", unlabeled_sub)
    row["claim_status"] = "unlabeled patches - not attributable to any crop"
    crop_rows.append(row)

sr_metrics_by_crop_df = (pd.DataFrame(crop_rows)
                         .sort_values("test_patches", ascending=False).reset_index(drop=True))
sr_metrics_by_crop_df.to_csv(RESULTS_DIR / "sr_metrics_by_crop.csv", index=False)
display(sr_metrics_by_crop_df.round(4))

# ---- classify every target crop for the manuscript guardrails ----------------------
SR_TEST_CROP_COUNTS = {c: int(len(observed.get(c, []))) for c in TARGET_CROPS}
sr_absent_crops     = [c for c in TARGET_CROPS if SR_TEST_CROP_COUNTS[c] == 0]
sr_thin_crops       = [c for c in TARGET_CROPS if 0 < SR_TEST_CROP_COUNTS[c] < MIN_TEST_CROP_COUNT]
sr_reportable_crops = [c for c in TARGET_CROPS if SR_TEST_CROP_COUNTS[c] >= MIN_TEST_CROP_COUNT]

print("\nTest-set composition by crop:")
for c in TARGET_CROPS:
    print(f"  {c:<8}: {SR_TEST_CROP_COUNTS[c]:>5} test patches"
          + ("   <-- ABSENT" if SR_TEST_CROP_COUNTS[c] == 0 else ""))
print(f"  total   : {len(test_patch_metrics_df):>5} test patches")

# ---- guardrail lines reused verbatim by the final summary --------------------------
sr_crop_claim_lines = []
if sr_reportable_crops:
    sr_crop_claim_lines.append(
        "Per-crop SR results may be reported ONLY for "
        + ", ".join(c.lower() for c in sr_reportable_crops)
        + " (" + ", ".join(f"{c.lower()} n={SR_TEST_CROP_COUNTS[c]}" for c in sr_reportable_crops)
        + "). All other SR results are aggregate agricultural-patch results.")
if sr_thin_crops:
    sr_crop_claim_lines.append(
        f"Crops with fewer than {MIN_TEST_CROP_COUNT} test patches "
        + ", ".join(f"{c.lower()} n={SR_TEST_CROP_COUNTS[c]}" for c in sr_thin_crops)
        + " are reported individually in results/sr_metrics_by_crop.csv and must not be "
          "described as equally demonstrated.")
if sr_absent_crops:
    sr_crop_claim_lines.append(
        "ABSENT from the SR test set entirely: "
        + ", ".join(c.lower() for c in sr_absent_crops)
        + ". SR performance was NOT demonstrated for "
        + ", ".join(c.lower() for c in sr_absent_crops)
        + " and no SR claim - aggregate or per-crop - covers "
        + ("them" if len(sr_absent_crops) > 1 else "it")
        + ". These crops remain valid in the BASELINE NDVI analysis, which is computed from "
          "CDL-masked scene means and does not depend on patches.")

for line in sr_crop_claim_lines:
    print("\nGUARDRAIL:", line)

undeclared_absent = [c for c in sr_absent_crops if c not in SR_CROPS_EXCLUDED_FROM_CLAIMS]
if undeclared_absent:
    print(f"\nACTION REQUIRED: {undeclared_absent} have zero test patches but are not listed in "
          "SR_CROPS_EXCLUDED_FROM_CLAIMS. The readiness check in Step 8 will fail until they "
          "are added there and removed from the SR claims in the manuscript.")

run_manifest["sr_test_crop_counts"] = SR_TEST_CROP_COUNTS
run_manifest["sr_crops_absent_from_test"] = sr_absent_crops
run_manifest["sr_crops_thin_in_test"] = sr_thin_crops
run_manifest["sr_crops_reportable"] = sr_reportable_crops
save_manifest()

In [ ]:
# Item 16: qualitative comparison from the held-out TEST set, with one fixed display
# stretch applied identically to all four panels and A-D panel labels.
examples = test_examples
fig, axes = plt.subplots(len(examples), 4, figsize=(12, 3.1*len(examples)))
if len(examples) == 1:
    axes = np.array([axes])

panel_titles = ["(A) LR input 32x32", "(B) Bicubic x4", "(C) Model prediction", "(D) HR target 128x128"]
for r, ex in enumerate(examples):
    for c, key in enumerate(["lr", "bicubic", "model", "hr"]):
        axes[r, c].imshow(to_display_rgb(chw_to_hwc(ex[key])),
                          interpolation="nearest" if key == "lr" else "none")
        if r == 0:
            axes[r, c].set_title(panel_titles[c], fontsize=10)
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
    axes[r, 0].set_ylabel(f"PSNR {ex['model_psnr']:.1f} dB", fontsize=8)

fig.suptitle(
    f"RGB = B4/B3/B2, fixed display stretch {DISPLAY_STRETCH} reflectance (visualisation only; "
    f"metrics use the unstretched arrays). Examples chosen by '{EXAMPLE_SELECTION_RULE}'.",
    fontsize=9, y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "figure_sr_visual_comparison.png", dpi=250, bbox_inches="tight"); plt.show()
print("Saved:", FIG_DIR / "figure_sr_visual_comparison.png")

In [ ]:
# Items 17 + 18: NDVI comparison with a SHARED colour scale and colorbar across the three
# NDVI panels, and ONE fixed error scale (0 to NDVI_ERR_VMAX) across every example, so
# residual magnitudes are comparable between rows instead of being rescaled per panel.
def tensor_to_ndvi_2d(x_chw):
    nd = compute_ndvi_tensor(x_chw.unsqueeze(0))
    return None if nd is None else nd.squeeze().detach().cpu().numpy()

if IN_CHANNELS <= max(RED_IDX, NIR_IDX):
    print("Not enough channels to compute NDVI - skipping NDVI impact figure.")
else:
    fig, axes = plt.subplots(len(examples), 4, figsize=(13, 3.0*len(examples)))
    if len(examples) == 1:
        axes = np.array([axes])

    ndvi_im = err_im = None
    ptitles = ["(A) NDVI bicubic", "(B) NDVI model", "(C) NDVI HR target",
               "(D) |model - HR| NDVI error"]
    for r, ex in enumerate(examples):
        ndvi_bic = tensor_to_ndvi_2d(ex["bicubic"])
        ndvi_mod = tensor_to_ndvi_2d(ex["model"])
        ndvi_hr  = tensor_to_ndvi_2d(ex["hr"])
        ndvi_err = np.abs(ndvi_mod - ndvi_hr)
        for c, panel in enumerate([ndvi_bic, ndvi_mod, ndvi_hr]):
            ndvi_im = axes[r, c].imshow(panel, vmin=NDVI_VMIN, vmax=NDVI_VMAX, cmap="RdYlGn")
            if r == 0:
                axes[r, c].set_title(ptitles[c], fontsize=10)
            axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        err_im = axes[r, 3].imshow(ndvi_err, vmin=0.0, vmax=NDVI_ERR_VMAX, cmap="magma")
        if r == 0:
            axes[r, 3].set_title(ptitles[3], fontsize=10)
        axes[r, 3].set_xticks([]); axes[r, 3].set_yticks([])
        axes[r, 0].set_ylabel(f"PSNR {ex['model_psnr']:.1f} dB", fontsize=8)
        frac_over = float((ndvi_err > NDVI_ERR_VMAX).mean())
        if frac_over > 0.01:
            print(f"   note: {frac_over*100:.1f}% of error pixels in {ex['file']} exceed the "
                  f"fixed display max {NDVI_ERR_VMAX} and are shown clipped.")

    fig.subplots_adjust(right=0.88)
    cb1 = fig.colorbar(ndvi_im, ax=axes[:, :3], fraction=0.02, pad=0.02)
    cb1.set_label(f"NDVI ({NDVI_VMIN} to {NDVI_VMAX}, shared across panels A-C)", fontsize=8)
    cb2 = fig.colorbar(err_im, ax=axes[:, 3], fraction=0.05, pad=0.02)
    cb2.set_label(f"absolute NDVI error (0 to {NDVI_ERR_VMAX}, fixed across all examples)", fontsize=8)

    fig.savefig(FIG_DIR / "figure_ndvi_impact.png", dpi=250, bbox_inches="tight"); plt.show()
    print("Saved:", FIG_DIR / "figure_ndvi_impact.png")
    print(f"NDVI RMSE (TEST set) - bicubic: {ndvi_b:.4f} | model: {ndvi_m:.4f} | "
          f"{'model better' if ndvi_m < ndvi_b else 'bicubic better'}")
    print("NOTE: this NDVI RMSE is computed over the WHOLE patch, not only crop-masked "
          "pixels. State that in the manuscript.")

Step 8: output checklist

In [ ]:
# Item: a real artifact/readiness check, replacing the previous two-string paper_ready test.
REQUIRED_ARTIFACTS = {
    "run manifest":                RESULTS_DIR / "run_manifest.json",
    "precipitation labels":        RESULTS_DIR / "annual_precipitation_labels.csv",
    "baseline NDVI (all crops)":   RESULTS_DIR / "baseline_ndvi_all_crops.csv",
    "baseline NDVI summary":       RESULTS_DIR / "baseline_ndvi_summary.csv",
    "baseline NDVI annual means":  RESULTS_DIR / "baseline_ndvi_annual_means.csv",
    "data audit":                  RESULTS_DIR / "data_audit.csv",
    "manifest details":            RESULTS_DIR / "manifest_details.csv",
    "dataset summary":             RESULTS_DIR / "dataset_summary.csv",
    "patch counts by year/crop":   RESULTS_DIR / "patch_counts_by_year_crop.csv",
    "train split":                 TRAIN_CSV,
    "val split":                   VAL_CSV,
    "test split":                  TEST_CSV,
    "training history":            RESULTS_DIR / "sr_training_history.csv",
    "per-patch test metrics":      RESULTS_DIR / "test_patch_metrics.csv",
    "bicubic vs model summary":    RESULTS_DIR / "sr_bicubic_vs_model_metrics.csv",
    "per-crop SR metrics":         RESULTS_DIR / "sr_metrics_by_crop.csv",
    "figure example selection":    RESULTS_DIR / "figure_example_patches.csv",
    "model checkpoint":            CHECKPOINT_DIR / "best_sr_model.pt",
    "baseline NDVI figure":        FIG_DIR / "figure_baseline_ndvi_by_crop.png",
    "training curves figure":      FIG_DIR / "figure_sr_training_curves.png",
    "SR comparison figure":        FIG_DIR / "figure_sr_visual_comparison.png",
    "NDVI impact figure":          FIG_DIR / "figure_ndvi_impact.png",
}

problems = []
for label, path in REQUIRED_ARTIFACTS.items():
    if not Path(path).exists():
        problems.append(f"missing artifact: {label} ({path})")

# audit integrity
if "lr_matches_4x_avg_hr" not in audit_df.columns:
    problems.append("data_audit.csv has no lr_match column - rerun the audit cell")
elif not bool(audit_df["valid"].astype(bool).all()):
    n_bad = int((~audit_df["valid"].astype(bool)).sum())
    problems.append(f"{n_bad} patches fail the audit and were excluded - report this count in Methods")

# provenance
if not (("real" in str(run_manifest.get("baseline_data_source"))) or
        ("cached_csv" in str(run_manifest.get("baseline_data_source")))):
    problems.append("baseline_data_source is not a real/cached source")
if not (("real" in str(run_manifest.get("sr_pair_data_source"))) or
        ("cached" in str(run_manifest.get("sr_pair_data_source")))):
    problems.append("sr_pair_data_source is not a real/cached source")
if run_manifest.get("split_strategy_used") not in {"temporal", "spatial_block"}:
    problems.append("split_strategy_used is not recorded in the manifest")
if run_manifest.get("audit_lr_match_enforced_in_validity") is not True:
    problems.append("audit validity does not enforce lr_match")

# crop-coverage honesty: any target crop absent from the SR test set must be declared
# excluded from the SR claims, otherwise the manuscript is free to imply it was evaluated.
for _c in sr_absent_crops:
    if _c not in SR_CROPS_EXCLUDED_FROM_CLAIMS:
        problems.append(
            f"{_c} has ZERO SR test patches but is not in SR_CROPS_EXCLUDED_FROM_CLAIMS - "
            "SR performance cannot be claimed for it; add it to the exclusion list and "
            "remove it from every SR claim in the manuscript")
_stale = [c for c in SR_CROPS_EXCLUDED_FROM_CLAIMS if c in sr_reportable_crops]
if _stale:
    print(f"NOTE: {_stale} are listed as excluded from SR claims but do have adequate test "
          "data. If that exclusion is stale, remove it so the results are not understated.")

paper_ready = (len(problems) == 0)
run_manifest["paper_ready"] = bool(paper_ready)
run_manifest["readiness_problems"] = problems
save_manifest()

print("=" * 70)
if paper_ready:
    print("READY: every required artifact exists and the audit passes.")
else:
    print(f"NOT READY - {len(problems)} problem(s):")
    for p in problems:
        print("  -", p)
print("=" * 70)

# ---------- results snippets, written from the frozen values only ----------
strongest = None
sig = baseline_summary.dropna(subset=["percent_change"])
if len(sig):
    strongest = sig.iloc[sig["percent_change"].abs().argmax()]

n_train = run_manifest.get("num_train_patches_used")
n_val   = run_manifest.get("num_val_patches_used")
n_test  = run_manifest.get("num_test_patches_used")
strategy_used = run_manifest.get("split_strategy_used")
patch_km = PATCH_SIZE_HR * HR_SCALE_METERS / 1000.0

# Crop-coverage guardrails come from the classification in Step 7 and report absent crops
# explicitly. The previous version could print "every target crop has at least the minimum
# number of test patches" while a crop was missing from the dataset altogether.
crop_coverage_lines = list(sr_crop_claim_lines)
if not crop_coverage_lines:
    crop_coverage_lines = ["No target crop reached the reporting threshold; report aggregate "
                           "agricultural-patch results only."]
crop_counts_line = ("Test-set composition: "
                    + ", ".join(f"{c.lower()} n={SR_TEST_CROP_COUNTS[c]}" for c in TARGET_CROPS)
                    + f", total n={len(test_patch_metrics_df)}.")

if strategy_used == "temporal":
    split_line = (f"A chronological holdout was used to prevent same-composite leakage between "
                  f"nearby or overlapping ~{patch_km:.2f} km patches drawn from the same "
                  f"growing-season composite: train = {SPLIT_TRAIN_YEARS}, validation = "
                  f"{SPLIT_VAL_YEAR}, test = {SPLIT_TEST_YEAR}. Because the same geographic "
                  "fields may recur across years, this tests temporal generalisation within "
                  "one region rather than complete spatial independence.")
else:
    under = run_manifest.get("split_underrepresented_crops_in_test_year")
    split_line = (f"A chronological split was rejected because {under} were nearly absent from "
                  f"{SPLIT_TEST_YEAR}. Whole ~{GEO_BLOCK_SIZE_DEG} deg geographic blocks "
                  "(never individual patches) were assigned to train/validation/test, so "
                  "nearby or overlapping patches cannot straddle the split. ALL manuscript "
                  "patch counts must be updated to match this strategy.")

lines = [
    "# Generated Results Snippets",
    "",
    "## Provenance",
    f"- Baseline NDVI source: {run_manifest['baseline_data_source']}",
    f"- SR pair source: {run_manifest['sr_pair_data_source']}",
    f"- Precipitation-group source: {run_manifest.get('year_labels_source')} "
    "(results/annual_precipitation_labels.csv)",
    f"- Paper-ready: {run_manifest['paper_ready']}",
    f"- Detected SR scale: x{DETECTED_SCALE} | channels: {IN_CHANNELS} | bands: {SR_BANDS}",
    f"- Valid patches: {run_manifest.get('audit_n_valid_patches')} of "
    f"{run_manifest.get('num_pair_files')} (lr_match enforced)",
    f"- Train/val/test patches used: {n_train}/{n_val}/{n_test}",
    f"- Seed: {SEED} | device: {device}",
    "",
    "## Methods sentences (copy these; they match the code)",
    f"Sentinel-2 scenes were filtered to CLOUDY_PIXEL_PERCENTAGE <= {MAX_CLOUD} and cloud-masked "
    f"with SCL classes {list(KEEP_SCL)} retained. Class 7 is unclassified/low-probability cloud "
    "and was kept to preserve coverage; residual contamination is disclosed as a limitation.",
    f"For each year, a cloud-masked {COMPOSITE_START_MONTHDAY} to {COMPOSITE_END_MONTHDAY} median "
    "Sentinel-2 composite was generated, and HR patches were extracted from these growing-season "
    "composites. Patches were not sampled year-round.",
    "Sentinel-2 scenes passing the cloud-filtering criteria were processed individually for the "
    "baseline; for each scene, mean NDVI was calculated over pixels assigned to the corresponding "
    f"crop class within the AOI, reduced at a {NDVI_REDUCE_SCALE} m analytical scale (B4 and B8 "
    "originate from native 10 m bands).",
    f"A crop-coverage threshold of {MIN_CROP_FRACTION} was applied, meaning at least "
    f"{MIN_CROP_FRACTION*100:.0f}% of CDL pixels in a candidate patch had to belong collectively "
    "to one of the target crop classes; the dominant crop was recorded separately.",
    f"Years with annual PRISM precipitation above the {YEARS[0]}-{YEARS[-1]} study-period mean were "
    "classified as higher-precipitation years, and years below it as lower-precipitation years. "
    "This is a relative grouping, not a climatological drought classification.",
    "",
    "## Split strategy",
    f"Strategy actually used: **{strategy_used}**. " + split_line,
    f"Resulting sizes: {n_train} train / {n_val} validation / {n_test} test patches.",
    "",
    "## Dataset composition",
    f"The dataset is dominated by {dominant_crop} ({dominant_share*100:.1f}% of all "
    f"{len(manifest_details_df)} patches); see results/patch_counts_by_year_crop.csv.",
    "",
    "## Baseline NDVI result (descriptive only - no inferential tests)",
]
if strongest is not None:
    lines.append(
        f"Across crops, {strongest['crop'].title()} showed the largest difference between "
        f"precipitation groups ({strongest['lower_minus_higher']:+.4f} NDVI, "
        f"{strongest['percent_change']:+.1f}%), with n = {int(strongest['n_higher_precip'])} "
        f"higher-precipitation and n = {int(strongest['n_lower_precip'])} lower-precipitation "
        "scene-observations. Full per-crop values are in results/baseline_ndvi_summary.csv. "
        "Welch t-tests were removed: repeated scene-level observations from one AOI are "
        "temporally autocorrelated and are not independent replicates.")
else:
    lines.append("See results/baseline_ndvi_summary.csv for per-crop values.")
lines += [
    "",
    f"## Super-resolution performance (TEST set, n={len(test_patch_metrics_df)} patches, "
    "per-patch paired evaluation)",
    f"Bicubic PSNR {psnr_b:.3f} dB vs model PSNR {psnr_m:.3f} dB. NDVI RMSE bicubic {ndvi_b:.4f} "
    f"vs model {ndvi_m:.4f}. See results/sr_bicubic_vs_model_metrics.csv for SD, median, 95% CI "
    "and the paired model-minus-bicubic difference.",
    crop_counts_line,
    *crop_coverage_lines,
    f"Qualitative examples were selected by the rule '{EXAMPLE_SELECTION_RULE}' "
    "(results/figure_example_patches.csv).",
    "",
    "## Wording guardrails",
    "- Report these as reconstruction-fidelity results, not crop-stress detection.",
    "- The SR results are AGGREGATE agricultural-patch results. Where broken down by crop, "
    "report only: " + (", ".join(c.lower() for c in sr_reportable_crops) if sr_reportable_crops
                       else "(none reached the threshold)") + ".",
    "- Do NOT claim SR performance for " + (", ".join(c.lower() for c in sr_absent_crops)
                                            if sr_absent_crops else "(no absent crops)")
    + ": zero test patches, so the model was never evaluated on "
    + ("them" if len(sr_absent_crops) != 1 else "it") + ".",
    "- Absent crops remain valid in the BASELINE NDVI analysis; that analysis uses "
    "CDL-masked scene means and does not depend on SR patches. Keep the two sections' "
    "crop lists separate and say so explicitly.",
    "- Say higher-/lower-precipitation, never wet/drought.",
    "- Say TEST set, not validation set, for every final number.",
    "- The model has SIX residual blocks.",
    "- LR inputs are idealised 4x area-averaged degradations of the same native 10 m "
    "Sentinel-2 target, so this is not validation against independent sub-10 m imagery.",
]
snippet = "\n".join([ln for ln in lines if ln != ""])
with open(PAPER_DIR / "generated_results_snippets.md", "w") as f:
    f.write(snippet)
print(snippet)

Step 9: freeze artifacts for the repository

Copies every frozen CSV, split file, checkpoint and figure into `frozen_artifacts/`, and writes `manuscript_reconciliation.csv` so every reported number can be traced to a file.

In [ ]:
# Item 10: copy every frozen artifact into one directory that can be committed alongside
# the notebook, so a reader can check every manuscript number without re-running hours of
# Earth Engine downloads and training.
import shutil

FROZEN_RESULTS = FROZEN_DIR / "results"
FROZEN_SPLITS  = FROZEN_DIR / "splits"
FROZEN_FIGS    = FROZEN_DIR / "figures"
FROZEN_CKPT    = FROZEN_DIR / "checkpoints"
for d in [FROZEN_RESULTS, FROZEN_SPLITS, FROZEN_FIGS, FROZEN_CKPT]:
    d.mkdir(parents=True, exist_ok=True)

TO_FREEZE = [
    (RESULTS_DIR / "run_manifest.json",                FROZEN_RESULTS),
    (RESULTS_DIR / "annual_precipitation_labels.csv",  FROZEN_RESULTS),
    (RESULTS_DIR / "baseline_ndvi_all_crops.csv",      FROZEN_RESULTS),
    (RESULTS_DIR / "baseline_ndvi_summary.csv",        FROZEN_RESULTS),
    (RESULTS_DIR / "baseline_ndvi_annual_means.csv",   FROZEN_RESULTS),
    (RESULTS_DIR / "baseline_ndvi_annual_summary.csv", FROZEN_RESULTS),
    (RESULTS_DIR / "data_audit.csv",                   FROZEN_RESULTS),
    (RESULTS_DIR / "manifest_details.csv",             FROZEN_RESULTS),
    (RESULTS_DIR / "dataset_summary.csv",              FROZEN_RESULTS),
    (RESULTS_DIR / "patch_counts_by_year_crop.csv",    FROZEN_RESULTS),
    (RESULTS_DIR / "sr_training_history.csv",          FROZEN_RESULTS),
    (RESULTS_DIR / "test_patch_metrics.csv",           FROZEN_RESULTS),
    (RESULTS_DIR / "sr_bicubic_vs_model_metrics.csv",  FROZEN_RESULTS),
    (RESULTS_DIR / "sr_metrics_by_crop.csv",           FROZEN_RESULTS),
    (RESULTS_DIR / "figure_example_patches.csv",       FROZEN_RESULTS),
    (TRAIN_CSV, FROZEN_SPLITS), (VAL_CSV, FROZEN_SPLITS), (TEST_CSV, FROZEN_SPLITS),
    (CHECKPOINT_DIR / "best_sr_model.pt",              FROZEN_CKPT),
    (FIG_DIR / "figure_baseline_ndvi_by_crop.png",     FROZEN_FIGS),
    (FIG_DIR / "figure_sr_training_curves.png",        FROZEN_FIGS),
    (FIG_DIR / "figure_sr_visual_comparison.png",      FROZEN_FIGS),
    (FIG_DIR / "figure_ndvi_impact.png",               FROZEN_FIGS),
    (FIG_DIR / "figure_split_map.png",                 FROZEN_FIGS),
    (PROJECT_ROOT / "requirements.txt",                FROZEN_DIR),
    (PAPER_DIR / "generated_results_snippets.md",      FROZEN_DIR),
]

copied, missing = [], []
for src, dest_dir in TO_FREEZE:
    src = Path(src)
    if src.exists():
        shutil.copy2(src, dest_dir / src.name)
        copied.append(src.name)
    else:
        missing.append(str(src))

# a manuscript-facing reconciliation table: every headline number and where it came from
recon = []
for metric in METRIC_NAMES:
    b = test_metrics_summary_df[(test_metrics_summary_df.metric == metric) &
                                (test_metrics_summary_df.method == "bicubic")]
    m = test_metrics_summary_df[(test_metrics_summary_df.metric == metric) &
                                (test_metrics_summary_df.method == "model")]
    d = test_metrics_summary_df[(test_metrics_summary_df.metric == metric) &
                                (test_metrics_summary_df.method == "model_minus_bicubic_paired")]
    if len(b) and len(m) and len(d):
        pct = 100 * float(d["mean"].iloc[0]) / float(b["mean"].iloc[0]) if float(b["mean"].iloc[0]) else np.nan
        recon.append({
            "metric": metric, "n_test_patches": int(m["n"].iloc[0]),
            "bicubic_mean": float(b["mean"].iloc[0]), "bicubic_sd": float(b["std"].iloc[0]),
            "model_mean": float(m["mean"].iloc[0]), "model_sd": float(m["std"].iloc[0]),
            "paired_delta_mean": float(d["mean"].iloc[0]),
            "paired_delta_ci95_low": float(d["ci95_low"].iloc[0]),
            "paired_delta_ci95_high": float(d["ci95_high"].iloc[0]),
            "percent_change_vs_bicubic": pct,
            "source_file": "results/sr_bicubic_vs_model_metrics.csv",
        })
reconciliation_df = pd.DataFrame(recon)
reconciliation_df.to_csv(FROZEN_DIR / "manuscript_reconciliation.csv", index=False)
display(reconciliation_df.round(5))

print(f"\nFroze {len(copied)} artifacts into {FROZEN_DIR}")
if missing:
    print(f"MISSING ({len(missing)}) - run the cells that produce these before committing:")
    for m_ in missing:
        print("   -", m_)
else:
    print("All expected artifacts present.")
print("\nEvery Table 2 number in the manuscript must match manuscript_reconciliation.csv.")
print("Do NOT commit the .npz pair files, your EE project ID, or old checkpoints.")